# SynerPlant TFM — feasibility-аудит v5.1: **manifest-aligned ABIOTIC + BIOTIC + supplementary**

Этот notebook отвечает на практический вопрос перед полной extraction:

> **Хватает ли текущего training/development корпуса для расширенного SynerPlant scope, и где остаются реальные evidence gaps?**

### Scope v5

Для feasibility исследования томата класс `A` анализируется иерархически:

- **ABIOTIC**: `drought`, `salinity`, `heat`, `cold`;
- **BIOTIC**: **все болезни pooled** как один domain для основной оценки достаточности;
- **Fusarium**: основной биотический subgroup, который показывается отдельно, но другие редкие болезни **не обязаны** достигать самостоятельных ML-порогов.

Точные disease/pathogen labels сохраняются в study-level output. В v5 исправлено различение `Rhizoctonia damping-off` vs `Pythium`, `Clavibacter` vs `Ralstonia`, а также комбинированных `early and late blight` и `bacterial speck and spot`. Добавлены `Verticillium` и nematode/root-knot как biotic metadata.

### Manifest alignment — lightweight Database V3

v5.1 проверен против облегчённой текущей структуры `TFM_ML_DATABASE_V3` (database layer 3.3.0) и **не полагается на старое число 707 studies**. Числа считываются динамически из Google Sheet.

Notebook ожидает канонические таблицы `studies`, `files`, `screening` и использует `study_id` как ключ. Он отдельно проверяет:

- соответствие `studies ↔ screening ↔ files`;
- наличие полного текста;
- supplement registry;
- текущую database/corpus version из `_README`;
- year governance для development corpus.

### Важное правило по времени

Для development-feasibility используются исследования **до 2023 включительно**. Если в manifest уже присутствует запись с известным `year >= 2024`, v5 **не считает её в TRAIN_DEV feasibility**, а выводит отдельный governance flag. Такая уже просмотренная статья также не считается чистым будущим external test; настоящий 2024+ test должен быть собран как unseen set после model/preprocessing freeze.

Строки с отсутствующим `year` не исключаются автоматически, чтобы не потерять большую часть исторического manifest, но их количество отдельно сообщается.

### Что сохранено

1. Stress/biostimulant/disease identity берётся из `screening` и безопасного fallback по Methods/Experimental design/Treatments — не из Introduction/Discussion/References.
2. Full-text structural signals (`SB vs S`, `C/S/B/SB`, variance, n) ищутся только в **Methods + Results**.
3. Biotic designs распознаются как обычная схема `C / S(pathogen) / B / SB(pathogen+biostimulant)`.
4. Все tabular supplementary (`.xlsx/.xls/.csv/.tsv`) анализируются отдельно и могут повысить extractability, но **никогда не увеличивают число независимых studies**.
5. Рассчитывается supplement uplift.
6. Для disease outcomes отдельно распознаются severity/incidence/pathogen load/lesion/survival endpoints.
7. `cold` и `BIOTIC pooled` входят в расширенную feasibility-оценку. Оригинальные пороги Table 4 сохранены и явно отделены от operational extension.

### Кэш

v5 повторно использует full-text text cache из v4/v3/v2. Supplementary используют отдельный v5 cache.

### Важно

Это всё ещё **feasibility audit**, а не финальная quantitative extraction. Любой `candidate` остаётся кандидатом до проверки treatment arms и чисел.


### Lightweight-schema compatibility

Notebook сначала читает metadata Google Sheet и загружает только реально существующие tabs. Удалённые из облегчённой базы служебные tabs (например, `run_log`/`qa_audit`) не считаются ошибкой и не требуются для feasibility. Канонические обязательные источники: `studies`, `files`, `screening`; остальные используются только если присутствуют.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab setup. No OpenAI / ChatGPT API is required.
%pip -q install pandas numpy matplotlib pyarrow PyMuPDF python-docx google-api-python-client tqdm openpyxl xlrd


In [ ]:
from google.colab import auth, drive as colab_drive
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError

import io, os, re, json, math, gzip, hashlib, time, csv
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymupdf
from docx import Document
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.set_option('display.max_colwidth', 160)


## 1. Конфигурация и требования, зафиксированные в TFM

Пороги ниже взяты из **Table 4 TFM**. Они относятся к независимым исследованиям и effect sizes, а не к количеству строк после размножения timepoints/endpoints.

Важное правило: если точное число effects ещё неизвестно, notebook **не подменяет его числом статей**. Такой пункт получает статус `PENDING_EXTRACTION`.


In [ ]:
# ---------- PROJECT CONFIG ----------
SPREADSHEET_ID = "1cD_iZVLF4hMeeAP7_DkaoT7YtjMvThn2k1WzwgwvR_g"  # TFM_ML_DATABASE_V3
PROJECT_DIR = Path("/content/drive/MyDrive/TFM_CORPUS_NORMALIZED")
AUDIT_DIR = PROJECT_DIR / "07_feasibility_audit_v5_1_light_manifest_aligned"
CACHE_DIR = AUDIT_DIR / "fulltext_text_cache"
LEGACY_CACHE_DIRS = [
    (PROJECT_DIR / "07_feasibility_audit_v4_abiotic_biotic" / "fulltext_text_cache", "CACHE_V4_REUSED"),
    (PROJECT_DIR / "07_feasibility_audit_v3_supplement_aware" / "fulltext_text_cache", "CACHE_V3_REUSED"),
    (PROJECT_DIR / "07_feasibility_audit_v2" / "fulltext_text_cache", "CACHE_V2_REUSED"),
]
SUPP_CACHE_DIR = AUDIT_DIR / "supplement_audit_cache"
FIG_DIR = AUDIT_DIR / "figures"
EXPORT_DIR = AUDIT_DIR / "exports"

# Full-text structural scan is needed for analytical classes only.
FULLTEXT_CLASSES = ["A","B","C","D"]
SUPPLEMENT_SCAN_CLASSES = ["A","B","C","D","E","X"]
MAX_STUDIES = None
FORCE_REPARSE = False
FORCE_SUPPLEMENT_REPARSE = False
MAX_PAGES_PER_PDF = None
MAX_SUPP_ROWS_PER_SHEET = 10000
MAX_SUPP_COLS_PER_SHEET = 250
MAX_SUPP_TEXT_CELLS = 30000

# Development corpus governance
TRAIN_DEV_MAX_YEAR = 2023
EXCLUDE_KNOWN_POST_CUTOFF_FROM_FEASIBILITY = True

# ---------- FROZEN TFM SUFFICIENCY RULES (Table 4) ----------
TFM_THRESHOLDS = {
    "multistress_min_studies": 40,
    "multistress_min_primary_effects": 100,
    "multistress_min_studies_per_main_stress": 8,
    "stress_specific_min_studies": 25,
    "stress_specific_min_effects": 60,
    "sxb_prediction_min_factorial_studies": 40,
    "sxb_prediction_min_interaction_effects": 100,
    "axb_prediction_min_strict_designs": 40,
    "interaction_pooling_min_studies": 5,
    "temporal_min_studies": 10,
    "temporal_min_primary_effects": 30,
    "temporal_min_studies_per_stress": 5,
    "applicability_low_evidence_below_studies": 5,
}

AUDIT_GUIDES = {
    "manual_review_if_heuristic_gap": True,
    "review_mining_secondary_sources_min": 1,
    "class_parse_coverage_warn_below": 0.90,
}

# Original TFM scope is retained for audit traceability.
ORIGINAL_TFM_STRESSES = ["drought", "salinity", "heat"]

# Current expanded scope requested for the updated project.
MAIN_ABIOTIC_STRESSES = ["drought", "salinity", "heat", "cold"]
MODEL_SCOPE_CATEGORIES = MAIN_ABIOTIC_STRESSES + ["biotic_all"]
FUSARIUM_SUBGROUP = "fusarium"

for p in [AUDIT_DIR, CACHE_DIR, SUPP_CACHE_DIR, FIG_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Audit directory:", AUDIT_DIR)
print("Original TFM stresses:", ORIGINAL_TFM_STRESSES)
print("Expanded scope:", MODEL_SCOPE_CATEGORIES, "+ Fusarium subgroup")
print("TFM thresholds:")
print(json.dumps(TFM_THRESHOLDS, indent=2, ensure_ascii=False))


### 1.1. Матрица ролей классов

Эта матрица предотвращает главную ошибку feasibility-аудита: **B/C/D/E не могут искусственно увеличивать размер основной A-модели**.


In [ ]:
CLASS_ROLE = pd.DataFrame([
    ["A", "Tomato + stress + biostimulant", "PRIMARY", True,  True,  True,  True,  "Main direct-mitigation / SxB evidence"],
    ["B", "Tomato + stress; no biostimulant", "STRESS_BASELINE", False, True,  False, False, "Auxiliary stress-response baseline; never substitutes A arms"],
    ["C", "Tomato + biostimulant; no stress", "BIOSTIM_BASELINE", False, True,  False, False, "Auxiliary biostimulant baseline; never substitutes A arms"],
    ["D", "Other crop + stress + biostimulant", "TRANSFER", False, True,  True,  True,  "Transfer/OOD evidence; does not increase tomato N"],
    ["E", "Other crop + only stress OR only biostimulant", "AUXILIARY", False, False, False, False, "Ontology/evidence-map/context only"],
    ["X", "Not eligible for primary effect-size modeling", "NON_TRAINING_SECONDARY", False, False, False, False, "Reviews/meta-analyses may support reference mining; no duplicate training rows"],
], columns=["class_final","definition","role","counts_for_core_tomato_ML","useful_for_evidence_map","useful_for_transfer","eligible_for_direct_effect_pipeline","notes"])
display(CLASS_ROLE)


## 2. Подключение Google Drive и чтение V3


In [ ]:
auth.authenticate_user()
colab_drive.mount('/content/drive', force_remount=False)

SCOPES = [
    "https://www.googleapis.com/auth/drive.readonly",
    "https://www.googleapis.com/auth/spreadsheets.readonly",
]
creds, _ = google.auth.default(scopes=SCOPES)
drive_api = build("drive", "v3", credentials=creds, cache_discovery=False)
sheets_api = build("sheets", "v4", credentials=creds, cache_discovery=False)
print("Google services ready.")


In [ ]:
def read_sheet_df(sheet_name, cell_range="A:AZ"):
    values = (
        sheets_api.spreadsheets().values().get(
            spreadsheetId=SPREADSHEET_ID,
            range=f"'{sheet_name}'!{cell_range}"
        ).execute().get("values", [])
    )
    if not values:
        return pd.DataFrame()
    header = [str(x).strip() for x in values[0]]
    rows = [list(r) + [""] * (len(header) - len(r)) for r in values[1:]]
    return pd.DataFrame([r[:len(header)] for r in rows], columns=header)

# Lightweight Database V3 may intentionally omit nonessential audit/log tabs.
# Resolve the live tab inventory first; only read tabs that actually exist.
meta_live = sheets_api.spreadsheets().get(
    spreadsheetId=SPREADSHEET_ID,
    fields="properties.title,sheets.properties(title,sheetId,gridProperties)"
).execute()
AVAILABLE_SHEETS = {x["properties"]["title"] for x in meta_live.get("sheets", [])}
print("Spreadsheet:", meta_live.get("properties", {}).get("title", ""))
print("Available tabs:", len(AVAILABLE_SHEETS))

SHEETS_TO_READ = [
    "_README","model_lock","external_test_registry","corpus_versions","studies","files",
    "screening","screening_taxonomy","extraction_queue","experiments","arms","measurements",
    "contrasts","ml_splits","dataset_registry","issues","data_assets",
    "risk_of_bias","study_relations"
]
OPTIONAL_LEGACY_TABS = ["run_log","qa_audit"]

# Hard requirement for the feasibility audit.
REQUIRED_TABS = {"_README","studies","files","screening"}
missing_required_tabs = sorted(REQUIRED_TABS - AVAILABLE_SHEETS)
if missing_required_tabs:
    raise RuntimeError(f"Current manifest is missing required tabs: {missing_required_tabs}")

DB = {}
for s in SHEETS_TO_READ:
    if s not in AVAILABLE_SHEETS:
        DB[s] = pd.DataFrame()
        print(f"{s:24s}: OPTIONAL / NOT PRESENT")
        continue
    DB[s] = read_sheet_df(s)
    print(f"{s:24s}: {len(DB[s]):5d} rows")

# Keep legacy keys available so downstream code never depends on their physical presence.
for s in OPTIONAL_LEGACY_TABS:
    DB[s] = read_sheet_df(s) if s in AVAILABLE_SHEETS else pd.DataFrame()

studies = DB["studies"]
files = DB["files"]
screening = DB["screening"]
queue = DB.get("extraction_queue", pd.DataFrame())

required_screening = {"study_id","class_final","screening_evidence","notes","has_fulltext","has_supplement","has_machine_readable_supplement"}
missing = sorted(required_screening - set(screening.columns))
if missing:
    raise RuntimeError(f"screening is missing required columns: {missing}")

# ---------- MANIFEST COMPATIBILITY CHECKS ----------
required_studies = {"study_id","year","dataset_role","screening_class_final"}
required_files = {"file_key","study_id","role","active_file_id","active_file_name","mime_type"}
missing_studies = sorted(required_studies - set(studies.columns))
missing_files = sorted(required_files - set(files.columns))
if missing_studies:
    raise RuntimeError(f"studies is missing required columns: {missing_studies}")
if missing_files:
    raise RuntimeError(f"files is missing required columns: {missing_files}")

# _README is key-value rather than a normal rectangular data table, so read it raw.
readme_values = (
    sheets_api.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID, range="'_README'!A1:B30"
    ).execute().get("values", [])
)
readme_map = {str(r[0]).strip(): str(r[1]).strip() for r in readme_values if len(r) >= 2}
DB_LAYER_VERSION = readme_map.get("TFM ML database layer", "UNKNOWN")
CORPUS_VERSION = readme_map.get("Corpus version", "UNKNOWN")
print("Manifest database version:", DB_LAYER_VERSION)
print("Manifest corpus version:", CORPUS_VERSION)

# Explicit compatibility marker for the currently verified lightweight manifest.
if DB_LAYER_VERSION == "Version 3.3.0":
    print("Compatibility: VERIFIED lightweight Database V3 schema")
else:
    print("Compatibility: schema checks passed; database version differs from the currently verified 3.3.0 snapshot")


## 3. Integrity snapshot и governance checks

Эта часть отвечает на два разных вопроса:

- **данных мало?** — проблема literature coverage;
- **данные есть, но pipeline ещё не готов?** — проблема extraction/QA, которую нельзя ошибочно решать новым поиском литературы.


In [ ]:
def nonempty_rows(df, key=None):
    if df.empty: return 0
    if key and key in df.columns:
        return int(df[key].astype(str).str.strip().ne("").sum())
    return len(df)

class_counts = screening["class_final"].astype(str).value_counts().reindex(["A","B","C","D","E","X"]).fillna(0).astype(int)

corpus_summary = pd.DataFrame({
    "metric": [
        "studies_total","fulltext_files","supplement_files","files_total","extraction_jobs",
        "experiments_extracted","arms_extracted","measurements_extracted","contrasts_derived",
        "external_test_registry_rows","model_lock_rows"
    ] + [f"class_{x}" for x in ["A","B","C","D","E","X"]],
    "value": [
        nonempty_rows(screening,"study_id"),
        int(files.get("role",pd.Series(dtype=str)).astype(str).str.upper().eq("FULLTEXT").sum()),
        int(files.get("role",pd.Series(dtype=str)).astype(str).str.upper().eq("SUPPLEMENT").sum()),
        len(files), nonempty_rows(queue, queue.columns[0] if len(queue.columns) else None),
        nonempty_rows(DB["experiments"],"experiment_id"), nonempty_rows(DB["arms"],"arm_id"),
        nonempty_rows(DB["measurements"],"measurement_id"), nonempty_rows(DB["contrasts"],"contrast_id"),
        nonempty_rows(DB["external_test_registry"], DB["external_test_registry"].columns[0] if len(DB["external_test_registry"].columns) else None),
        nonempty_rows(DB["model_lock"], DB["model_lock"].columns[0] if len(DB["model_lock"].columns) else None),
        *[int(class_counts.get(x,0)) for x in ["A","B","C","D","E","X"]],
    ]
})
display(corpus_summary)

# Current dataset-role governance
role_counts = {}
if "dataset_role" in studies.columns:
    role_counts = studies["dataset_role"].replace("",np.nan).dropna().value_counts().to_dict()
print("Dataset roles:", role_counts)
print("External test rows:", nonempty_rows(DB["external_test_registry"], DB["external_test_registry"].columns[0] if len(DB["external_test_registry"].columns) else None))


### 3.1. Protocol alignment: original TFM vs current V3 governance and expanded stress scope

Два изменения отслеживаются отдельно:

1. **Temporal validation**: текущий уже просмотренный корпус остаётся `TRAIN_DEV`; настоящий external test собирается позднее после model/preprocessing freeze.
2. **Scientific scope**: original TFM explicitly focused on drought/salinity/high temperature. Текущий расширенный audit дополнительно оценивает `cold` и `BIOTIC pooled`, а Fusarium показывает как subgroup. Это operational extension; если оно остаётся в финальной TFM, соответствующие Objective/Methods нужно будет обновить текстово.


In [ ]:
# Governance checks: dataset role + temporal cutoff
current_train_dev_only = bool(role_counts) and set(role_counts).issubset({"TRAIN_DEV"})
external_rows = nonempty_rows(DB["external_test_registry"], DB["external_test_registry"].columns[0] if len(DB["external_test_registry"].columns) else None)

study_year_num = pd.to_numeric(studies.get("year", pd.Series(index=studies.index,dtype=object)), errors="coerce")
known_post_cutoff_mask = study_year_num.gt(TRAIN_DEV_MAX_YEAR)
year_unknown_mask = study_year_num.isna() & studies["study_id"].astype(str).str.strip().ne("")
known_post_cutoff = studies.loc[known_post_cutoff_mask, [c for c in ["study_id","doi","title","year","screening_class_final","dataset_role"] if c in studies.columns]].copy()
known_post_cutoff["governance_flag"] = "KNOWN_POST_2023_ALREADY_SEEN__EXCLUDE_FROM_TRAIN_DEV_FEASIBILITY"

print(f"Known publication year > {TRAIN_DEV_MAX_YEAR} in current manifest:", len(known_post_cutoff))
print("Studies with missing publication year:", int(year_unknown_mask.sum()))
if len(known_post_cutoff):
    display(known_post_cutoff)

protocol_alignment = pd.DataFrame([
    {
        "topic":"temporal_validation",
        "tfm_requirement":"Evaluate temporal generalization / external holdout",
        "current_manifest_state": f"dataset_role counts={role_counts}; external_test_registry={external_rows}",
        "status":"PENDING_EXTERNAL_TEST" if external_rows==0 else "TEST_REGISTERED",
        "implication":"The confirmatory 2024+ test must be collected as unseen evidence after model/preprocessing freeze."
    },
    {
        "topic":"known_post_2023_inside_current_manifest",
        "tfm_requirement":f"TRAIN_DEV feasibility cutoff <= {TRAIN_DEV_MAX_YEAR}",
        "current_manifest_state": f"known year > {TRAIN_DEV_MAX_YEAR}: {len(known_post_cutoff)}",
        "status":"EXCLUDED_FROM_V5_FEASIBILITY" if len(known_post_cutoff) else "ALIGNED",
        "implication":"Already-seen post-2023 records are not counted in development feasibility and should not be reused as a pristine confirmatory test."
    },
    {
        "topic":"publication_year_completeness",
        "tfm_requirement":"Publication year should be known for strict temporal governance",
        "current_manifest_state": f"missing year={int(year_unknown_mask.sum())}",
        "status":"METADATA_GAP__DO_NOT_AUTO_EXCLUDE" if int(year_unknown_mask.sum()) else "COMPLETE",
        "implication":"Unknown-year records remain provisionally eligible because automatic exclusion would discard a large fraction of the frozen corpus; resolve year metadata before final temporal lock."
    },
    {
        "topic":"current_corpus_role",
        "tfm_requirement":"No leakage between development and confirmatory evaluation",
        "current_manifest_state":"Current reviewed corpus is marked TRAIN_DEV" if current_train_dev_only else str(role_counts),
        "status":"ROLE_FIELD_ALIGNED__YEAR_GUARD_APPLIED_SEPARATELY" if current_train_dev_only else "REVIEW_ROLE_ASSIGNMENTS",
        "implication":"Dataset role alone is not used to override the <=2023 development cutoff."
    },
    {
        "topic":"expanded_stress_scope",
        "tfm_requirement":"Original TFM named drought/salinity/high temperature; current project also evaluates cold and pooled biotic disease stress",
        "current_manifest_state":"ABIOTIC=drought/salinity/heat/cold; BIOTIC=all diseases pooled; Fusarium subgroup reported separately",
        "status":"OPERATIONAL_EXTENSION__TFM_TEXT_UPDATE_NEEDED_IF_RETAINED",
        "implication":"Keep exact disease/pathogen metadata, but do not require each rare disease subtype to meet a standalone ML threshold."
    }
])
display(protocol_alignment)


## 4. Taxonomy: abiotic stress, biotic disease, biostimulant and outcomes

Классификация multi-label. Для болезней используется иерархия:

`BIOTIC pooled → disease subtype → pathogen metadata`.

Fusarium выводится отдельно как основной subgroup. Редкие disease types сохраняются, но не рассматриваются как обязательные самостоятельные predictive tasks.


In [ ]:
# Two-level stress patterns: screening text is curated and may use short labels such as "tomato + cold";
# Methods fallback is stricter to avoid incidental mentions.
STRESS_PATTERNS_SCREENING = {
    "salinity": r"\b(salinity|saline|salt stress|salt-stress|nacl|seawater|salinization|salinized)\b",
    "drought": r"\b(drought|water stress|water-stress|water deficit|water-deficit|deficit irrigation|irrigation deficit|soil drying|water limitation|water withholding)\b",
    "heat": r"\b(heat|heat stress|heat-stress|heat shock|high temperature|high-temperature|thermal stress|thermotoler\w*|heat[- ]induced|supra[- ]optimal temperature|heatwave|heat wave)\b",
    "cold": r"\b(cold|cold stress|cold-stress|cold[- ]induced|cold tolerance|low temperature|low-temperature|low night temperature|chill\w*|freez\w*|frost)\b",
    "osmotic": r"\b(osmotic stress|polyethylene glycol|peg[- ]induced)\b",
    "nutrient": r"\b(nutrient deficiency|nitrogen deficiency|low nitrogen|low n\b|phosphorus deficiency)\b",
    "toxic_metals": r"\b(cadmium|heavy metal|arsenic|lead|chromium|nickel|metal toxicity)\b",
    "flooding_hypoxia": r"\b(waterlogging|flooding|hypoxia|anoxia)\b",
}
STRESS_PATTERNS_METHODS = {
    **STRESS_PATTERNS_SCREENING,
    "heat": r"\b(heat stress|heat-stress|heat shock|heat treatment|high temperature|high-temperature|thermal stress|thermotoler\w*|heat[- ]induced|supra[- ]optimal temperature|heatwave|heat wave)\b",
    "cold": r"\b(cold stress|cold-stress|cold treatment|cold[- ]induced|cold tolerance|low temperature|low-temperature|low night temperature|chill\w*|freez\w*|frost)\b",
}

# Named pathogens / unambiguous disease agents. Syndrome-only phrases are handled separately below.
DISEASE_PATTERNS = {
    "fusarium": r"\b(fusarium(?:\s+oxysporum)?|f\.\s*oxysporum)\b",
    "ralstonia_bacterial_wilt": r"\bralstonia(?:\s+solanacearum)?\b",
    "alternaria_early_blight": r"\balternaria(?:\s+solani)?\b",
    "phytophthora_late_blight": r"\bphytophthora(?:\s+infestans)?\b",
    "botrytis_gray_mold": r"\bbotrytis(?:\s+cinerea)?\b",
    "rhizoctonia_root_rot": r"\brhizoctonia(?:\s+solani)?\b",
    "pythium_damping_off": r"\bpythium(?:\s+aphanidermatum)?\b",
    "xanthomonas_bacterial_spot": r"\bxanthomonas(?:\s+euvesicatoria)?\b",
    "pseudomonas_bacterial_speck": r"\bpseudomonas\s+syringae(?:\s+pv\.?\s+tomato)?\b",
    "tylcv": r"\b(tomato yellow leaf curl virus|tylcv)\b",
    "clavibacter_bacterial_canker": r"\bclavibacter(?:\s+michiganensis)?\b",
    "verticillium_wilt": r"\bverticillium(?:\s+dahliae|\s+albo[- ]atrum)?\b",
    "nematode_root_disease": r"\b(root[- ]knot nematod\w*|nematod\w*|meloidogyne\w*)\b",
}
DISEASE_GENERIC_RX = re.compile(
    r"\b(pathogen(?:ic)?|infected|infection|biotic disease|disease severity|disease incidence|soilborne pathogen|root rot|wilt disease|blight disease|damping[- ]off|biocontrol of .{0,40}(?:disease|wilt|rot|blight))\b",
    re.I,
)

BIOSTIM_PATTERNS = {
    "PGPR_bacteria": r"\b(pgpr|rhizobacter\w*|bacillus\w*|pseudomonas(?!\s+syringae\b)\w*|azospirillum\w*|azotobacter\w*|paenibacillus\w*|actinobacter\w*|plant growth[- ]promoting bacter\w*)\b",
    "AMF_mycorrhiza": r"\b(mycorrhiz\w*|amf\b|rhizophagus\w*|glomus\w*|septoglomus\w*)\b",
    "beneficial_fungi": r"\b(trichoderma\w*|endophytic fung\w*|fungal endophyte\w*|penicillium\w*)\b",
    "seaweed_algae": r"\b(seaweed\w*|algae\w*|algal\w*|ascophyllum\w*|ecklonia\w*|ulva\w*|chlorella\w*|microalgae\w*|cyanobacter\w*)\b",
    "protein_hydrolysate_amino": r"\b(amino acid\w*|protein hydrolysate\w*|hydrolysate\w*)\b",
    "humic_fulvic": r"\b(humic\w*|fulvic\w*|humate\w*)\b",
    "silicon": r"\b(silicon|silicate|sio2|orthosilicic acid)\b",
    "chitosan": r"\bchitosan\b",
    "selenium": r"\b(selenium|selenite|selenate)\b",
    "biochar_compost": r"\b(biochar|compost\w*|vermicompost\w*|compost tea|leachate)\b",
    "plant_extracts": r"\b(moringa extract|moringa leaf extract|plant extract|botanical extract|licorice extract|garlic extract)\b",
    "signaling_regulators": r"\b(salicylic acid|abscisic acid|nitric oxide|melatonin|jasmonic acid|methyl jasmonate|meja\b|brassinosteroid\w*|24[- ]?epibrassinolide|5-aminolevulinic acid|aminolevulinic acid|polyamine\w*|spermidine|putrescine|gamma[- ]aminobutyric acid|gaba\b|da-6\b|karrikin\w*)\b",
    "nano_materials": r"\b(nano\w*|nanoparticle\w*|nano-nutrient\w*)\b",
}

OUTCOME_PATTERNS = {
    "dry_biomass": r"\b(dry biomass|dry weight|shoot dry weight|root dry weight|plant dry weight|dry matter)\b",
    "fresh_biomass": r"\b(fresh biomass|fresh weight|shoot fresh weight|root fresh weight)\b",
    "yield": r"\b(yield|fruit yield|marketable yield|fruit weight|fruit number|number of fruits|productivity)\b",
    "growth": r"\b(plant height|leaf area|root length|shoot length|stem diameter|growth rate|biomass)\b",
    "photosynthesis_gas_exchange": r"\b(photosynthesis|photosynthetic rate|net co2 assimilation|gas exchange|stomatal conductance|transpiration)\b",
    "chlorophyll_fluorescence": r"\b(fv/fm|chlorophyll fluorescence|photochemical|npq|photosystem ii|psii)\b",
    "water_relations": r"\b(relative water content|rwc|water potential|leaf water potential|hydraulic conductance)\b",
    "ions_nutrients": r"\b(na\+|k\+|na/k|sodium|potassium|ion homeostasis|nutrient uptake|mineral content)\b",
    "oxidative_damage": r"\b(mda|malondialdehyde|electrolyte leakage|hydrogen peroxide|h2o2|reactive oxygen|ros)\b",
    "antioxidants": r"\b(catalase|cat activity|superoxide dismutase|sod activity|peroxidase|ascorbate peroxidase|glutathione reductase|antioxidant enzyme)\b",
    "osmolytes": r"\b(proline|soluble sugar|glycine betaine|osmolyte)\b",
    "fruit_quality": r"\b(brix|soluble solids|fruit quality|ascorbic acid|vitamin c|carotenoid|lycopene|phenolic|flavonoid)\b",
    "disease_severity": r"\b(disease severity|severity index|disease index|wilt index|disease rating|symptom severity)\b",
    "disease_incidence": r"\b(disease incidence|incidence of disease|percent(?:age)? diseased|infected plants|wilt incidence)\b",
    "pathogen_load": r"\b(pathogen load|pathogen biomass|fungal biomass|colony forming units|cfu|pathogen population|viral titre|virus titer|qPCR.{0,30}(?:pathogen|fung|virus))\b",
    "lesion_symptoms": r"\b(lesion size|lesion diameter|lesion area|necrotic area|symptom score)\b",
    "survival_mortality": r"\b(survival|mortality|dead plants|plant death)\b",
}

COMPILED_STRESS_SCREENING = {k: re.compile(v, re.I) for k,v in STRESS_PATTERNS_SCREENING.items()}
COMPILED_STRESS_METHODS = {k: re.compile(v, re.I) for k,v in STRESS_PATTERNS_METHODS.items()}
# Alias used by supplementary/header parsing where short factor labels are expected.
COMPILED_STRESS = COMPILED_STRESS_SCREENING
COMPILED_DISEASE = {k: re.compile(v, re.I) for k,v in DISEASE_PATTERNS.items()}
COMPILED_BIO = {k: re.compile(v, re.I) for k,v in BIOSTIM_PATTERNS.items()}
COMPILED_OUTCOME = {k: re.compile(v, re.I) for k,v in OUTCOME_PATTERNS.items()}

def screening_text(row):
    fields = [
        row.get("title",""), row.get("screening_evidence",""), row.get("notes",""),
        row.get("stress_types_hint",""), row.get("biostimulant_types_hint","")
    ]
    return " ".join(str(x or "") for x in fields)

def detect_labels(text, compiled):
    return [k for k,rx in compiled.items() if rx.search(str(text or ""))]

def detect_disease_labels(text):
    """Pathogen-first disease labeling with syndrome disambiguation."""
    text = str(text or "")
    labels = detect_labels(text, COMPILED_DISEASE)

    def add(label):
        if label not in labels:
            labels.append(label)

    # Explicit combined syndromes used in the current manifest.
    if re.search(r"\bearly\s+(?:and|&)\s+late\s+blight\b", text, re.I):
        add("alternaria_early_blight"); add("phytophthora_late_blight")
    if re.search(r"\bbacterial\s+speck\s+(?:and|&)\s+spot\b", text, re.I):
        add("pseudomonas_bacterial_speck"); add("xanthomonas_bacterial_spot")

    # Syndrome-only fallback when the syndrome is biologically specific enough.
    if re.search(r"\bearly\s+blight\b", text, re.I): add("alternaria_early_blight")
    if re.search(r"\blate\s+blight\b", text, re.I): add("phytophthora_late_blight")
    if re.search(r"\b(?:gray|grey)\s+(?:mold|mould)\b", text, re.I): add("botrytis_gray_mold")
    if re.search(r"\bbacterial\s+spot\b", text, re.I): add("xanthomonas_bacterial_spot")
    if re.search(r"\bbacterial\s+speck\b", text, re.I): add("pseudomonas_bacterial_speck")

    # Do NOT map generic damping-off to Pythium: Rhizoctonia can also cause damping-off.
    # Do NOT map generic bacterial wilt to Ralstonia when another named pathogen is present.
    named_conflict = any(x in labels for x in [
        "fusarium","clavibacter_bacterial_canker","verticillium_wilt",
        "xanthomonas_bacterial_spot","pseudomonas_bacterial_speck"
    ])
    if re.search(r"\bbacterial\s+wilt\b", text, re.I) and not named_conflict:
        add("ralstonia_bacterial_wilt")

    if not labels and DISEASE_GENERIC_RX.search(text):
        labels = ["other_biotic_disease"]
    return list(dict.fromkeys(labels))

def infer_stress_domains(abiotic_labels, disease_labels):
    out=[]
    if abiotic_labels: out.append("ABIOTIC")
    if disease_labels: out.append("BIOTIC")
    return out

def domain_group(domains):
    s=set(domains or [])
    if s=={"ABIOTIC"}: return "ABIOTIC_ONLY"
    if s=={"BIOTIC"}: return "BIOTIC_ONLY"
    if {"ABIOTIC","BIOTIC"}.issubset(s): return "ABIOTIC+BIOTIC"
    return "UNRESOLVED"

scr = screening.copy()
scr = scr[scr["study_id"].astype(str).str.strip().ne("")].copy()
scr["screening_text"] = scr.apply(screening_text, axis=1)
scr["stress_labels_screening"] = scr["screening_text"].map(lambda x: detect_labels(x, COMPILED_STRESS_SCREENING))
scr["disease_labels_screening"] = scr["screening_text"].map(detect_disease_labels)
scr["biostim_labels_screening"] = scr["screening_text"].map(lambda x: detect_labels(x, COMPILED_BIO))
scr["stress_domains_screening"] = scr.apply(lambda r: infer_stress_domains(r["stress_labels_screening"], r["disease_labels_screening"]), axis=1)

# Attach year / role from the canonical studies table and enforce the development cutoff.
study_meta = studies[[c for c in ["study_id","year","dataset_role","screening_class_final"] if c in studies.columns]].copy()
study_meta["study_id"] = study_meta["study_id"].astype(str)
study_meta = study_meta.rename(columns={"year":"publication_year_manifest","dataset_role":"dataset_role_manifest","screening_class_final":"class_final_studies"})
scr = scr.merge(study_meta, on="study_id", how="left")
scr["publication_year_numeric"] = pd.to_numeric(scr["publication_year_manifest"], errors="coerce")
scr["known_post_2023"] = scr["publication_year_numeric"].gt(TRAIN_DEV_MAX_YEAR)
scr["publication_year_unknown"] = scr["publication_year_numeric"].isna()
scr["dataset_role_manifest"] = scr["dataset_role_manifest"].fillna("").astype(str)
scr["analysis_eligible"] = (~scr["known_post_2023"]) if EXCLUDE_KNOWN_POST_CUTOFF_FROM_FEASIBILITY else True
scr["analysis_eligible"] = scr["analysis_eligible"] & scr["dataset_role_manifest"].isin(["", "TRAIN_DEV"])

# Manifest identity sanity flag: do not auto-exclude, but surface obvious crop-title contradictions.
NON_TOMATO_CROP_RX = re.compile(r"\b(arabidopsis|pepper|capsicum|potato|maize|corn|wheat|rice|lettuce|cucumber|soybean|barley)\b", re.I)
TOMATO_RX = re.compile(r"\b(tomato|solanum\s+lycopersicum|lycopersicon\s+esculentum|micro[- ]tom)\b", re.I)
scr["manifest_title_crop_mismatch_flag"] = scr.apply(
    lambda r: bool(str(r.get("class_final",""))=="A" and NON_TOMATO_CROP_RX.search(str(r.get("title","") or "")) and not TOMATO_RX.search(str(r.get("title","") or ""))), axis=1
)

scr_all = scr.copy()
year_governance = scr_all[[c for c in ["study_id","class_final","title","publication_year_manifest","publication_year_numeric","dataset_role_manifest","known_post_2023","publication_year_unknown","analysis_eligible"] if c in scr_all.columns]].copy()
manifest_identity_review = scr_all[scr_all["manifest_title_crop_mismatch_flag"]].copy()

print("Manifest studies in screening:", scr_all["study_id"].nunique())
print("Analysis-eligible studies after <=2023 known-year guard:", int(scr_all["analysis_eligible"].sum()))
print("Known post-2023 excluded from feasibility:", int(scr_all["known_post_2023"].sum()))
print("Unknown publication year retained provisionally:", int(scr_all["publication_year_unknown"].sum()))
print("Class-A title/crop identity flags:", int(scr_all["manifest_title_crop_mismatch_flag"].sum()))

# All downstream feasibility calculations use analysis-eligible records only.
scr = scr_all[scr_all["analysis_eligible"]].copy()
display(scr.groupby("class_final")["study_id"].nunique().reindex(["A","B","C","D","E","X"]).to_frame("analysis_eligible_studies"))


## 5. Full-text feasibility scan: **Methods/Results only**

V2 давал ложные stress/biostimulant labels, потому что искал ключевые слова по Introduction, Discussion и References.

В v3:

- идентичность stress/biostimulant берётся из `screening`;
- если screening не определил категорию, fallback допускается только из **Methods / Materials and Methods / Experimental design / Treatments**;
- признаки treatment arms, factorial design, variance и `n` ищутся в Methods + Results;
- Introduction/Discussion/References не участвуют в этих решениях.

Это всё ещё эвристика: она создаёт **очередь кандидатов**, а не финальные effect-size records.


In [ ]:
# One FULLTEXT record per study
fulltext = files[files["role"].astype(str).str.upper().eq("FULLTEXT")].copy()
fulltext = fulltext[fulltext["study_id"].astype(str).str.strip().ne("")].copy()
fulltext = fulltext.drop_duplicates("study_id", keep="first")
fulltext_map = fulltext.set_index(fulltext["study_id"].astype(str)).to_dict("index")
print("FULLTEXT studies registered:", len(fulltext_map))

def safe_cache_name(study_id):
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(study_id))
    return s[:100] + ".txt.gz"

def download_drive_file(file_id):
    request = drive_api.files().get_media(fileId=file_id, supportsAllDrives=True)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request, chunksize=4*1024*1024)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return fh.getvalue()

def source_bytes(row, role):
    """Prefer the mounted Drive copy; fall back to Drive API."""
    name = str(row.get("active_file_name") or row.get("source_file_name") or "").strip()
    subdir = "01_fulltext" if role == "FULLTEXT" else "02_supplements"
    if name:
        p = PROJECT_DIR / subdir / name
        if p.exists() and p.is_file():
            return p.read_bytes(), "LOCAL_DRIVE_MOUNT"
    file_id = str(row.get("active_file_id") or row.get("source_file_id") or "").strip()
    if not file_id:
        raise FileNotFoundError("No active/source file id")
    return download_drive_file(file_id), "DRIVE_API"

def pdf_to_text(pdf_bytes, max_pages=None):
    doc = pymupdf.open(stream=pdf_bytes, filetype="pdf")
    try:
        n = doc.page_count if max_pages is None else min(doc.page_count, max_pages)
        pages=[]
        for pno in range(n):
            try: pages.append(doc[pno].get_text("text", sort=True) or "")
            except Exception: pages.append("")
        return "\n".join(pages), doc.page_count
    finally:
        doc.close()

def docx_to_text(docx_bytes):
    bio = io.BytesIO(docx_bytes)
    doc = Document(bio)
    parts=[p.text for p in doc.paragraphs]
    for table in doc.tables:
        for row in table.rows:
            parts.append("\t".join(c.text for c in row.cells))
    return "\n".join(parts)

def get_study_text(study_id, row):
    cache_path = CACHE_DIR / safe_cache_name(study_id)
    if not FORCE_REPARSE:
        candidates=[(cache_path,"CACHE_V5")]
        candidates += [(d / safe_cache_name(study_id), label) for d,label in LEGACY_CACHE_DIRS]
        for cp, label in candidates:
            if cp.exists():
                with gzip.open(cp, "rt", encoding="utf-8", errors="ignore") as f:
                    return f.read(), label, None, label

    mime = str(row.get("mime_type") or "").strip().lower()
    name = str(row.get("active_file_name") or row.get("source_file_name") or "").strip().lower()
    try:
        data, route = source_bytes(row, "FULLTEXT")
        if data[:5] == b"%PDF-" or mime == "application/pdf" or name.endswith(".pdf"):
            text, pages = pdf_to_text(data, MAX_PAGES_PER_PDF)
        elif data[:2] == b"PK" and ("wordprocessingml" in mime or name.endswith(".docx")):
            text, pages = docx_to_text(data), None
        else:
            return "", f"UNSUPPORTED_MIME:{mime or name}", None, route
        if text.strip():
            with gzip.open(cache_path, "wt", encoding="utf-8") as f:
                f.write(text)
            return text, "PARSED", pages, route
        return "", "TEXT_EMPTY_OR_SCANNED", pages, route
    except HttpError as e:
        return "", f"DRIVE_HTTP_ERROR:{getattr(e,'status_code','')}", None, "DRIVE_API"
    except Exception as e:
        return "", f"PARSE_ERROR:{type(e).__name__}", None, "UNKNOWN"

# -------- Section isolation --------
METHOD_HEAD = re.compile(r"^\s*(materials?\s+and\s+methods?|methods?|experimental\s+(?:design|procedure|setup)|plant\s+materials?|growth\s+conditions?|treatments?|experimental\s+treatments?)\s*[:.]?\s*$", re.I)
RESULT_HEAD = re.compile(r"^\s*(results?|results?\s+and\s+discussion)\s*[:.]?\s*$", re.I)
DISCUSS_HEAD = re.compile(r"^\s*(discussion|conclusions?|references?|bibliography|acknowledg(?:e)?ments?)\s*[:.]?\s*$", re.I)
REF_HEAD = re.compile(r"^\s*(references?|bibliography)\s*[:.]?\s*$", re.I)
DESIGN_WINDOW_RX = re.compile(
    r"\b(experimental design|treatment(?:s)?|stress treatment|growth conditions|irrigation regime|nacl|water deficit|drought|heat stress|cold stress|cold|chilling|low temperature|biostimulant|inoculation|pathogen|infection|disease|fusarium|ralstonia|alternaria|phytophthora|botrytis|rhizoctonia|pythium|xanthomonas|clavibacter|verticillium|nematode|application rate|factorial)\b",
    re.I
)

def _heading_positions(lines, rx):
    return [i for i,line in enumerate(lines) if rx.match(line.strip())]

def isolate_sections(text):
    """Return methods, results and a methods+results analysis block without references."""
    lines=[str(x) for x in str(text or "").splitlines()]
    if not lines:
        return "", "", ""
    refs=_heading_positions(lines, REF_HEAD)
    ref_start=refs[0] if refs else len(lines)
    core=lines[:ref_start]

    meth=_heading_positions(core, METHOD_HEAD)
    res=_heading_positions(core, RESULT_HEAD)
    disc=_heading_positions(core, DISCUSS_HEAD)

    methods_text=""
    results_text=""
    if meth:
        ms=meth[0]
        candidates=[x for x in res+disc if x>ms]
        me=min(candidates) if candidates else min(len(core), ms+max(120, int(0.35*len(core))))
        methods_text="\n".join(core[ms:me])
    else:
        keep=[]
        for i,line in enumerate(core):
            if DESIGN_WINDOW_RX.search(line):
                keep.extend(range(max(0,i-5), min(len(core),i+12)))
        keep=sorted(set(keep))
        methods_text="\n".join(core[i] for i in keep)

    if res:
        rs=res[0]
        candidates=[x for x in disc if x>rs]
        re_=min(candidates) if candidates else len(core)
        results_text="\n".join(core[rs:re_])

    analysis_text=(methods_text+"\n"+results_text).strip()
    return methods_text, results_text, analysis_text


In [ ]:
# Conservative structural signals applied ONLY to Methods/Results blocks.
EXPOSURE_TERMS = r"(?:salinity|saline|nacl|drought|water deficit|water stress|heat stress|high temperature|cold stress|cold[- ]induced|low temperature|chill\w*|fusarium|ralstonia|alternaria|phytophthora|botrytis|rhizoctonia|pythium|xanthomonas|pseudomonas syringae|clavibacter|verticillium|nematode|meloidogyne|tylcv|pathogen|infected|infection|disease|wilt|blight|root rot|damping[- ]off|gray mold|grey mold)"
BIO_ACTION_TERMS = r"(?:biostimul|pgpr|rhizobacter|bacillus|trichoderma|mycorrhiz|amf|seaweed|extract|hydrolysate|chitosan|silicon|selenium|melatonin|salicylic acid|brassinosteroid|spermidine|putrescine|treated|treatment)"

RX = {
    "factorial": re.compile(r"\b(factorial design|factorial experiment|two[- ]factor|three[- ]factor|2\s*[x×]\s*2|randomized complete block.*factorial)\b", re.I),
    "with_without": re.compile(r"\b(with and without|with or without|inoculated and non[- ]inoculated|infected and non[- ]infected|treated and untreated|untreated and treated|non[- ]treated and treated|mock[- ]inoculated)\b", re.I),
    "nonstress_stress": re.compile(rf"\b(well[- ]watered.{{0,220}}(drought|water stress)|non[- ]saline.{{0,220}}salin|control.{{0,220}}(nacl|salinity|drought|heat stress|cold stress)|non[- ]stressed.{{0,220}}stressed|(healthy|non[- ]infected|uninfected|non[- ]inoculated|mock[- ]inoculated|pathogen[- ]free).{{0,220}}{EXPOSURE_TERMS})\b", re.I|re.S),
    "stress_only_comparison": re.compile(rf"\b(stressed control|stress control|infected control|disease control|pathogen control|pathogen[- ]inoculated control|untreated.{{0,180}}{EXPOSURE_TERMS}|{EXPOSURE_TERMS}.{{0,180}}without.{{0,100}}{BIO_ACTION_TERMS}|control plants.{{0,180}}{EXPOSURE_TERMS})\b", re.I|re.S),
    "bio_only_arm": re.compile(rf"\b(non[- ]stressed.{{0,220}}{BIO_ACTION_TERMS}|well[- ]watered.{{0,220}}{BIO_ACTION_TERMS}|(healthy|non[- ]infected|uninfected|mock[- ]inoculated).{{0,220}}{BIO_ACTION_TERMS}|biostimulant alone|beneficial inoculation alone|treatment alone)\b", re.I|re.S),
    "combined_arm": re.compile(rf"\b(({BIO_ACTION_TERMS}).{{0,160}}({EXPOSURE_TERMS})|({EXPOSURE_TERMS}).{{0,160}}({BIO_ACTION_TERMS}))\b", re.I|re.S),
    "variance": re.compile(r"\b(standard deviation|standard error|s\.d\.|s\.e\.|sem\b|confidence interval|error bars)\b|mean\s*[±]", re.I),
    "n_reported": re.compile(r"\b(n\s*=\s*\d+|\d+\s+(biological\s+)?replicates|replicated\s+\d+\s+times|\d+\s+plants\s+per\s+treatment|\d+\s+pots\s+per\s+treatment|\d+\s+plots\s+per\s+treatment)\b", re.I),
    "randomization": re.compile(r"\b(randomi[sz]ed|randomized complete block|completely randomized|split[- ]plot|block design)\b", re.I),
    "multi_dose": re.compile(r"\b(dose|concentration|rate)s?\b.{0,100}\b(\d+(?:\.\d+)?).{0,30}(\d+(?:\.\d+)?)", re.I|re.S),
    "timepoints": re.compile(r"\b(after|at)\s+\d+\s+(h|hours?|d|days?|weeks?)\b", re.I),
    "table": re.compile(r"\btable\s+\d+\b", re.I),
    "figure": re.compile(r"\bfig(?:ure)?\.?\s*\d+\b", re.I),
    "axb_language": re.compile(r"\b(synerg\w*|additiv\w*|antagon\w*|combined application|co[- ]application|combination treatment|alone and in combination|individually and in combination)\b", re.I),
    "axb_structure": re.compile(r"\b(control).{0,500}\b(alone|single).{0,500}\b(alone|single).{0,500}\b(combination|combined|co[- ]application)\b", re.I|re.S),
}
REVIEW_RX = re.compile(r"\b(systematic review|scoping review|narrative review|review article|meta[- ]analysis|meta[- ]analytic)\b", re.I)
ERRATUM_RX = re.compile(r"\b(erratum|corrigendum|correction to)\b", re.I)

def analyze_fulltext(study_id, class_final, title, screening_blob, screening_stress, screening_disease, screening_bio, text, parse_status, pages, read_route):
    methods_text, results_text, analysis_text = isolate_sections(text)
    design_low = methods_text.lower() if methods_text else ""
    analysis_low = analysis_text.lower() if analysis_text else ""

    stress_methods = detect_labels(design_low, COMPILED_STRESS_METHODS) if design_low else []
    disease_methods = detect_disease_labels(design_low) if design_low else []
    bio_methods = detect_labels(design_low, COMPILED_BIO) if design_low else []

    # Screening is preferred, but Methods can safely add a missing category because Intro/Discussion/References are excluded.
    stress_final = list(dict.fromkeys(list(screening_stress or []) + stress_methods))
    disease_final = list(dict.fromkeys(list(screening_disease or []) + disease_methods))
    bio_final = list(dict.fromkeys(list(screening_bio or []) + bio_methods))
    domains_final = infer_stress_domains(stress_final, disease_final)

    stress_source = "SCREENING+METHODS" if screening_stress and stress_methods else ("SCREENING" if screening_stress else ("METHODS_FALLBACK" if stress_methods else "UNRESOLVED"))
    disease_source = "SCREENING+METHODS" if screening_disease and disease_methods else ("SCREENING" if screening_disease else ("METHODS_FALLBACK" if disease_methods else "UNRESOLVED"))
    bio_source = "SCREENING+METHODS" if screening_bio and bio_methods else ("SCREENING" if screening_bio else ("METHODS_FALLBACK" if bio_methods else "UNRESOLVED"))

    outcome_labels = detect_labels(analysis_low, COMPILED_OUTCOME) if analysis_low else []
    sig = {k: bool(rx.search(analysis_low)) for k,rx in RX.items()} if analysis_low else {k:False for k in RX}

    has_stress = bool(stress_final or disease_final)
    has_bio = bool(bio_final)
    direct_candidate = bool(has_stress and has_bio and sig["combined_arm"] and (sig["stress_only_comparison"] or sig["factorial"] or sig["with_without"]))
    strict_candidate = bool(has_stress and has_bio and sig["combined_arm"] and sig["bio_only_arm"] and sig["nonstress_stress"] and (sig["factorial"] or sig["with_without"]))
    strict_possible = bool(has_stress and has_bio and sig["combined_arm"] and (sig["factorial"] or (sig["nonstress_stress"] and sig["bio_only_arm"])))
    stats_ready_proxy = bool(sig["variance"] and sig["n_reported"])

    stress_baseline_candidate = bool(class_final == "B" and has_stress and (sig["nonstress_stress"] or sig["stress_only_comparison"] or sig["factorial"]))
    biostim_baseline_candidate = bool(class_final == "C" and has_bio and (sig["with_without"] or sig["bio_only_arm"] or sig["factorial"]))
    transfer_direct_candidate = bool(class_final == "D" and direct_candidate)
    axb_candidate = bool(class_final in {"A","C","D","E"} and sig["axb_language"] and len(bio_final) >= 2)
    axb_strict_possible = bool(axb_candidate and (sig["factorial"] or sig["axb_structure"]))

    x_blob=(str(title or "")+" "+str(screening_blob or "")).lower()
    x_review = bool(class_final == "X" and REVIEW_RX.search(x_blob))
    x_meta = bool(class_final == "X" and re.search(r"\bmeta[- ]analysis|meta[- ]analytic\b", x_blob, re.I))
    x_erratum = bool(class_final == "X" and ERRATUM_RX.search(x_blob))

    return {
        "study_id": str(study_id), "class_final": class_final,
        "parse_status": parse_status, "read_route": read_route, "pages": pages, "text_chars": len(text or ""),
        "methods_chars": len(methods_text), "results_chars": len(results_text),
        "stress_labels_methods": ";".join(stress_methods),
        "disease_labels_methods": ";".join(disease_methods),
        "biostim_labels_methods": ";".join(bio_methods),
        "stress_labels_resolved": ";".join(stress_final),
        "disease_labels_resolved": ";".join(disease_final),
        "stress_domains_resolved": ";".join(domains_final),
        "biostim_labels_resolved": ";".join(bio_final),
        "stress_label_source": stress_source,
        "disease_label_source": disease_source,
        "biostim_label_source": bio_source,
        "outcome_labels_fulltext": ";".join(outcome_labels),
        "heuristic_direct_candidate_ft": direct_candidate,
        "heuristic_strict_sxb_candidate_ft": strict_candidate,
        "heuristic_strict_sxb_possible_ft": strict_possible,
        "heuristic_stress_baseline_candidate_ft": stress_baseline_candidate,
        "heuristic_biostim_baseline_candidate_ft": biostim_baseline_candidate,
        "heuristic_transfer_direct_candidate_ft": transfer_direct_candidate,
        "heuristic_axb_candidate_ft": axb_candidate,
        "heuristic_axb_strict_possible_ft": axb_strict_possible,
        "stats_variance_signal_ft": sig["variance"], "stats_n_signal_ft": sig["n_reported"],
        "stats_ready_proxy_ft": stats_ready_proxy, "randomization_signal": sig["randomization"],
        "multi_dose_signal_ft": sig["multi_dose"], "timepoint_signal_ft": sig["timepoints"],
        "x_review_secondary_source": x_review, "x_meta_analysis_secondary_source": x_meta,
        "x_erratum_or_correction": x_erratum,
        "table_count_text": len(RX["table"].findall(analysis_low)) if analysis_low else 0,
        "figure_count_text": len(RX["figure"].findall(analysis_low)) if analysis_low else 0,
        **{f"outcome_{k}_ft": (k in outcome_labels) for k in COMPILED_OUTCOME}
    }


In [ ]:
# RUN CONSERVATIVE FULL-TEXT FEASIBILITY SCAN
scan_source = scr[scr["class_final"].isin(FULLTEXT_CLASSES)].copy()
if MAX_STUDIES is not None:
    scan_source = scan_source.head(MAX_STUDIES)

records=[]
for _, r in tqdm(scan_source.iterrows(), total=len(scan_source), desc="Scanning Methods/Results"):
    sid = str(r["study_id"]).strip()
    frow = fulltext_map.get(sid)
    title = str(r.get("title","") or "")
    sblob = str(r.get("screening_text","") or "")
    sstress = r.get("stress_labels_screening",[]) if isinstance(r.get("stress_labels_screening",[]),list) else []
    sdisease = r.get("disease_labels_screening",[]) if isinstance(r.get("disease_labels_screening",[]),list) else []
    sbio = r.get("biostim_labels_screening",[]) if isinstance(r.get("biostim_labels_screening",[]),list) else []
    if frow is None:
        records.append(analyze_fulltext(sid, str(r.get("class_final","")), title, sblob, sstress, sdisease, sbio, "", "NO_FULLTEXT_ROW", None, "NONE"))
        continue
    text, status, pages, route = get_study_text(sid, frow)
    records.append(analyze_fulltext(sid, str(r.get("class_final","")), title, sblob, sstress, sdisease, sbio, text, status, pages, route))

ft_audit = pd.DataFrame(records)
print("Scanned:", len(ft_audit))
display(ft_audit.groupby(["class_final","parse_status"])["study_id"].count().rename("n").reset_index())

# X/E are not subjected to expensive structural scanning; retain screening-level taxonomy.
missing_sids=set(scr["study_id"].astype(str))-set(ft_audit.get("study_id",pd.Series(dtype=str)).astype(str))
light=[]
for _,r in scr[scr["study_id"].astype(str).isin(missing_sids)].iterrows():
    sid=str(r["study_id"])
    cl=str(r.get("class_final",""))
    blob=(str(r.get("title","") or "")+" "+str(r.get("screening_text","") or "")).lower()
    sstress=r.get("stress_labels_screening",[]) if isinstance(r.get("stress_labels_screening",[]),list) else []
    sdisease=r.get("disease_labels_screening",[]) if isinstance(r.get("disease_labels_screening",[]),list) else []
    sbio=r.get("biostim_labels_screening",[]) if isinstance(r.get("biostim_labels_screening",[]),list) else []
    light.append({
        "study_id":sid,"class_final":cl,"parse_status":"NOT_SCANNED_BY_DESIGN","read_route":"NONE",
        "stress_labels_resolved":";".join(sstress),
        "disease_labels_resolved":";".join(sdisease),
        "stress_domains_resolved":";".join(infer_stress_domains(sstress,sdisease)),
        "biostim_labels_resolved":";".join(sbio),
        "stress_label_source":"SCREENING" if sstress else "UNRESOLVED",
        "disease_label_source":"SCREENING" if sdisease else "UNRESOLVED",
        "biostim_label_source":"SCREENING" if sbio else "UNRESOLVED",
        "x_review_secondary_source":bool(cl=="X" and REVIEW_RX.search(blob)),
        "x_meta_analysis_secondary_source":bool(cl=="X" and re.search(r"\bmeta[- ]analysis|meta[- ]analytic\b",blob,re.I)),
        "x_erratum_or_correction":bool(cl=="X" and ERRATUM_RX.search(blob)),
    })
if light:
    ft_audit=pd.concat([ft_audit,pd.DataFrame(light)],ignore_index=True,sort=False)


## 6. Supplement-aware feasibility: `.xlsx/.xls/.csv/.tsv`

Supplementary parser v4 is disease-aware. Он распознаёт не только abiotic stress factors, но и pathogen/disease factors, включая Fusarium. Поэтому таблица с `healthy / pathogen / biostimulant / pathogen+biostimulant` может повысить study до direct или strict S×B candidate.


In [ ]:
MACHINE_MIMES = {
    "text/csv", "text/tab-separated-values",
    "application/vnd.ms-excel",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
}
MACHINE_EXTS = {".csv",".tsv",".xls",".xlsx"}

def _ext_from_row(row):
    name=str(row.get("active_file_name") or row.get("source_file_name") or "").strip().lower()
    return Path(name).suffix.lower()

def is_machine_supp(row):
    mime=str(row.get("mime_type") or "").strip().lower()
    return mime in MACHINE_MIMES or _ext_from_row(row) in MACHINE_EXTS

supp_files=files[files["role"].astype(str).str.upper().eq("SUPPLEMENT")].copy()
supp_files=supp_files[supp_files["study_id"].astype(str).str.strip().ne("")].copy()
supp_files["is_machine_tabular"]=supp_files.apply(is_machine_supp,axis=1)
supp_files["class_final"]=supp_files["study_id"].astype(str).map(scr.set_index(scr["study_id"].astype(str))["class_final"].to_dict())
supp_machine=supp_files[supp_files["is_machine_tabular"] & supp_files["class_final"].isin(SUPPLEMENT_SCAN_CLASSES)].copy()

print("All supplementary files:",len(supp_files))
print("Machine-readable tabular supplementary files:",len(supp_machine))
print("Studies with machine-readable tabular supplementary:",supp_machine["study_id"].astype(str).nunique())
display(supp_machine.groupby("class_final")["study_id"].nunique().rename("n_studies").reset_index())

CTRL_RX=re.compile(r"\b(control|untreated|mock|vehicle|well[- ]watered|non[- ]stressed|nonstressed|normal condition|without stress|healthy|non[- ]infected|uninfected|non[- ]inoculated|mock[- ]inoculated|pathogen[- ]free)\b",re.I)
VAR_HDR_RX=re.compile(r"(^|\b)(sd|s\.d\.|standard deviation|se|s\.e\.|sem|standard error|ci|confidence interval)(\b|$)",re.I)
N_HDR_RX=re.compile(r"(^|\b)(n|replicates?|biological replicates?|sample size)(\b|$)",re.I)
RAW_ID_RX=re.compile(r"\b(replicate|rep\.?|plant id|sample id|pot id|plot id|block|biological replicate)\b",re.I)
DOSE_RX=re.compile(r"\b(dose|concentration|rate|ppm|mg\s*l|g\s*l|ml\s*l|µm|um|mm|cfu)\b",re.I)
TIME_RX=re.compile(r"\b(time|timepoint|day|days|hour|hours|week|weeks|dat|dai)\b",re.I)
GROUP_HDR_RX=re.compile(r"\b(treatment|group|condition|stress|irrigation|inoculation|biostimulant|amendment|pathogen|disease|infection|isolate)\b",re.I)
EXACT_ARM_TOKEN_RX=re.compile(r"^(C|S|B|SB)$",re.I)

def dataframe_cells(df, max_cells=MAX_SUPP_TEXT_CELLS):
    vals=[]
    for row in df.itertuples(index=False,name=None):
        for v in row:
            if pd.isna(v): continue
            s=str(v).strip()
            if s and s.lower() not in {"nan","none"}:
                vals.append(s)
                if len(vals)>=max_cells: return vals
    return vals

def possible_factorial_grid(df):
    """Conservative signal for stress/pathogen factor × biostimulant factor with >=2 levels each."""
    if df.empty: return False
    nrows=min(12,len(df)); ncols=min(MAX_SUPP_COLS_PER_SHEET,df.shape[1])
    for hr in range(nrows):
        headers=[str(x).strip() if not pd.isna(x) else "" for x in df.iloc[hr,:ncols].tolist()]
        exposure_cols=[]; bio_cols=[]
        for j,h in enumerate(headers):
            hl=h.lower()
            if (detect_labels(hl,COMPILED_STRESS) or detect_disease_labels(hl) or
                re.search(r"\b(stress|salinity|nacl|irrigation|water regime|temperature|pathogen|disease|infection|fusarium|ralstonia|botrytis|rhizoctonia|pythium)\b",hl,re.I)):
                exposure_cols.append(j)
            if detect_labels(hl,COMPILED_BIO) or re.search(r"\b(biostimulant|beneficial inocul|treatment product|extract|amf|pgpr|trichoderma|bacillus)\b",hl,re.I):
                bio_cols.append(j)
        if not exposure_cols or not bio_cols: continue
        data=df.iloc[hr+1:,:ncols]
        for sc in exposure_cols:
            svals=data.iloc[:,sc].map(lambda x:str(x).strip().lower() if not pd.isna(x) else "")
            if len({x for x in svals.tolist() if x})<2: continue
            for bc in bio_cols:
                bvals=data.iloc[:,bc].map(lambda x:str(x).strip().lower() if not pd.isna(x) else "")
                if len({x for x in bvals.tolist() if x})<2: continue
                pairs={(s,b) for s,b in zip(svals,bvals) if s and b}
                if len(pairs)<4: continue
                s_to_b=defaultdict(set); b_to_s=defaultdict(set)
                for s,b in pairs:
                    s_to_b[s].add(b); b_to_s[b].add(s)
                if sum(len(v)>=2 for v in s_to_b.values())>=2 and sum(len(v)>=2 for v in b_to_s.values())>=2:
                    return True
    return False

def classify_label_cells(cells):
    ctrl=stress_only=bio_only=combined=False
    exact=set(); abiotic_found=set(); disease_found=set(); bio_found=set()
    for s in cells:
        if len(s)>250: continue
        st=detect_labels(s,COMPILED_STRESS)
        ds=detect_disease_labels(s)
        bs=detect_labels(s,COMPILED_BIO)
        abiotic_found.update(st); disease_found.update(ds); bio_found.update(bs)
        exposure=bool(st or ds)
        if CTRL_RX.search(s): ctrl=True
        if exposure and bs: combined=True
        elif exposure: stress_only=True
        elif bs: bio_only=True
        if EXACT_ARM_TOKEN_RX.match(s.strip()): exact.add(s.strip().upper())
    exact_four={"C","S","B","SB"}.issubset(exact)
    return ctrl,stress_only,bio_only,combined,exact_four,sorted(abiotic_found),sorted(disease_found),sorted(bio_found)

def read_tabular_supp_bytes(data,name):
    ext=Path(str(name)).suffix.lower(); sheets=[]
    if ext in {".xlsx",".xls"}:
        xf=pd.ExcelFile(io.BytesIO(data))
        for sh in xf.sheet_names:
            try:
                df=pd.read_excel(xf,sheet_name=sh,header=None,nrows=MAX_SUPP_ROWS_PER_SHEET)
                if df.shape[1]>MAX_SUPP_COLS_PER_SHEET: df=df.iloc[:,:MAX_SUPP_COLS_PER_SHEET]
                sheets.append((str(sh),df))
            except Exception:
                continue
    elif ext in {".csv",".tsv"}:
        raw=data.decode("utf-8-sig",errors="replace"); sep="\t" if ext==".tsv" else None
        try:
            df=pd.read_csv(io.StringIO(raw),header=None,sep=sep,engine="python",nrows=MAX_SUPP_ROWS_PER_SHEET)
        except Exception:
            df=pd.read_csv(io.StringIO(raw),header=None,sep="," if ext==".csv" else "\t",nrows=MAX_SUPP_ROWS_PER_SHEET)
        if df.shape[1]>MAX_SUPP_COLS_PER_SHEET: df=df.iloc[:,:MAX_SUPP_COLS_PER_SHEET]
        sheets.append(("table",df))
    return sheets

def supplement_cache_path(file_key):
    safe=re.sub(r"[^A-Za-z0-9_.-]+","_",str(file_key))[:120]
    return SUPP_CACHE_DIR/(safe+".json")

def analyze_supp_file(row):
    file_key=str(row.get("file_key") or row.get("active_file_id") or row.get("source_file_id") or "unknown")
    cp=supplement_cache_path(file_key)
    if cp.exists() and not FORCE_SUPPLEMENT_REPARSE:
        try: return json.load(open(cp,encoding="utf-8"))
        except Exception: pass
    sid=str(row.get("study_id") or "")
    name=str(row.get("active_file_name") or row.get("source_file_name") or "")
    out={"study_id":sid,"file_key":file_key,"file_name":name,"supp_parse_status":"UNREAD"}
    try:
        data,route=source_bytes(row,"SUPPLEMENT")
        tables=read_tabular_supp_bytes(data,name)
        out["supp_read_route"]=route; out["supp_sheet_count"]=len(tables)
        if not tables:
            out["supp_parse_status"]="NO_TABULAR_SHEETS"
        else:
            outcomes=set(); st=set(); ds=set(); bs=set()
            any_var=any_n=any_raw=any_dose=any_time=any_group=False
            any_direct=any_strict=any_factorial=False; n_data_rows=0
            for sh,df in tables:
                cells=dataframe_cells(df); joined=" | ".join(cells)
                outcomes.update(detect_labels(joined,COMPILED_OUTCOME))
                ctrl,so,bo,co,exact4,st_here,ds_here,bs_here=classify_label_cells(cells)
                st.update(st_here); ds.update(ds_here); bs.update(bs_here)
                any_var |= any(VAR_HDR_RX.search(x) for x in cells[:8000])
                any_n |= any(N_HDR_RX.search(x) for x in cells[:8000])
                any_raw |= any(RAW_ID_RX.search(x) for x in cells[:8000])
                any_dose |= any(DOSE_RX.search(x) for x in cells[:8000])
                any_time |= any(TIME_RX.search(x) for x in cells[:8000])
                any_group |= any(GROUP_HDR_RX.search(x) for x in cells[:8000])
                fac=possible_factorial_grid(df); any_factorial |= fac
                any_direct |= bool((co and so) or fac)
                any_strict |= bool(exact4 or (ctrl and so and bo and co) or fac)
                n_data_rows += int(df.dropna(how="all").shape[0])
            out.update({
                "supp_parse_status":"PARSED","supp_data_rows":n_data_rows,
                "supp_variance_signal":bool(any_var),"supp_n_signal":bool(any_n),
                "supp_raw_replicates_possible":bool(any_raw),"supp_dose_signal":bool(any_dose),
                "supp_time_signal":bool(any_time),"supp_group_structure_signal":bool(any_group),
                "supp_factorial_grid_signal":bool(any_factorial),
                "supp_direct_upgrade_signal":bool(any_direct),"supp_strict_sxb_upgrade_signal":bool(any_strict),
                "supp_stress_labels":";".join(sorted(st)),
                "supp_disease_labels":";".join(sorted(ds)),
                "supp_biostim_labels":";".join(sorted(bs)),
                "supp_outcome_labels":";".join(sorted(outcomes)),
            })
    except Exception as e:
        out["supp_parse_status"]=f"ERROR:{type(e).__name__}"; out["supp_error"]=str(e)[:300]
    with open(cp,"w",encoding="utf-8") as f:
        json.dump(out,f,ensure_ascii=False,indent=2)
    return out

supp_records=[]
for _,r in tqdm(supp_machine.iterrows(),total=len(supp_machine),desc="Scanning tabular supplementary"):
    supp_records.append(analyze_supp_file(r))

supp_file_audit=pd.DataFrame(supp_records)
if supp_file_audit.empty:
    supp_file_audit=pd.DataFrame(columns=["study_id","file_key","file_name","supp_parse_status"])

def semi_union(series):
    vals=[]
    for x in series.dropna().astype(str):
        for z in x.split(";"):
            z=z.strip()
            if z and z not in vals: vals.append(z)
    return ";".join(vals)

if len(supp_file_audit):
    # Ensure optional columns exist even after partial/error parses.
    for c in ["supp_variance_signal","supp_n_signal","supp_raw_replicates_possible","supp_dose_signal","supp_time_signal","supp_factorial_grid_signal","supp_direct_upgrade_signal","supp_strict_sxb_upgrade_signal"]:
        if c not in supp_file_audit: supp_file_audit[c]=False
    for c in ["supp_stress_labels","supp_disease_labels","supp_biostim_labels","supp_outcome_labels"]:
        if c not in supp_file_audit: supp_file_audit[c]=""
    if "supp_data_rows" not in supp_file_audit: supp_file_audit["supp_data_rows"]=0
    supp_study=supp_file_audit.groupby("study_id",as_index=False).agg(
        tabular_supp_files=("file_key","count"),
        parsed_tabular_supp_files=("supp_parse_status",lambda s:int((s=="PARSED").sum())),
        supp_variance_signal=("supp_variance_signal","max"), supp_n_signal=("supp_n_signal","max"),
        supp_raw_replicates_possible=("supp_raw_replicates_possible","max"), supp_dose_signal=("supp_dose_signal","max"),
        supp_time_signal=("supp_time_signal","max"), supp_factorial_grid_signal=("supp_factorial_grid_signal","max"),
        supp_direct_upgrade_signal=("supp_direct_upgrade_signal","max"), supp_strict_sxb_upgrade_signal=("supp_strict_sxb_upgrade_signal","max"),
        supp_stress_labels=("supp_stress_labels",semi_union), supp_disease_labels=("supp_disease_labels",semi_union),
        supp_biostim_labels=("supp_biostim_labels",semi_union), supp_outcome_labels=("supp_outcome_labels",semi_union),
        supp_data_rows=("supp_data_rows","sum"),
    )
else:
    supp_study=pd.DataFrame(columns=["study_id"])

display(supp_file_audit.head(30))
display(supp_study.head(30))


### 6.1. Supplement uplift

Ниже supplementary объединяются с full text **на уровне того же `study_id`**. Итоговые признаки `heuristic_*` используются в feasibility verdict; отдельные `*_ft` сохраняются, чтобы было видно реальный вклад supplementary.


In [ ]:
# Merge screening + conservative full-text features + supplement features.
ft_merge = ft_audit.drop(columns=["class_final"], errors="ignore")
audit = scr.merge(ft_merge, on="study_id", how="left")
audit = audit.merge(supp_study, on="study_id", how="left")

# Manifest provenance carried into every study-level audit row.
audit["manifest_db_version"] = DB_LAYER_VERSION
audit["manifest_corpus_version"] = CORPUS_VERSION

def split_labels(x):
    if isinstance(x,list): return x
    s=str(x or "").strip()
    return [z for z in s.split(";") if z]

def union_safe_labels(row, screen_col, resolved_col, supp_col):
    vals=[]
    screen=row.get(screen_col,[])
    if isinstance(screen,list): vals.extend(screen)
    vals.extend(split_labels(row.get(resolved_col,"")))
    vals.extend(split_labels(row.get(supp_col,"")))
    return list(dict.fromkeys([x for x in vals if x]))

audit["stress_labels"] = audit.apply(lambda r: union_safe_labels(r,"stress_labels_screening","stress_labels_resolved","supp_stress_labels"),axis=1)
audit["disease_labels"] = audit.apply(lambda r: union_safe_labels(r,"disease_labels_screening","disease_labels_resolved","supp_disease_labels"),axis=1)
audit["biostim_labels"] = audit.apply(lambda r: union_safe_labels(r,"biostim_labels_screening","biostim_labels_resolved","supp_biostim_labels"),axis=1)
audit["stress_domains"] = audit.apply(lambda r: infer_stress_domains(r["stress_labels"],r["disease_labels"]),axis=1)
audit["stress_domain_group"] = audit["stress_domains"].map(domain_group)
audit["is_abiotic"] = audit["stress_domains"].map(lambda x:"ABIOTIC" in x)
audit["is_biotic"] = audit["stress_domains"].map(lambda x:"BIOTIC" in x)
audit["is_fusarium"] = audit["disease_labels"].map(lambda x:"fusarium" in x)

audit["has_supplement_bool"] = audit["has_supplement"].astype(str).str.upper().eq("TRUE")
audit["has_machine_supp_bool"] = audit["has_machine_readable_supplement"].astype(str).str.upper().eq("TRUE")
audit["has_registered_tabular_supp"] = audit.get("tabular_supp_files",pd.Series(0,index=audit.index)).fillna(0).astype(float).gt(0)

bool_cols=[
    "heuristic_direct_candidate_ft","heuristic_strict_sxb_candidate_ft","heuristic_strict_sxb_possible_ft",
    "heuristic_stress_baseline_candidate_ft","heuristic_biostim_baseline_candidate_ft","heuristic_transfer_direct_candidate_ft",
    "heuristic_axb_candidate_ft","heuristic_axb_strict_possible_ft","stats_variance_signal_ft","stats_n_signal_ft","stats_ready_proxy_ft",
    "supp_variance_signal","supp_n_signal","supp_raw_replicates_possible","supp_direct_upgrade_signal","supp_strict_sxb_upgrade_signal",
    "supp_factorial_grid_signal"
]
for c in bool_cols:
    if c not in audit.columns: audit[c]=False
    audit[c]=audit[c].fillna(False).astype(bool)

audit["heuristic_direct_candidate"] = audit["heuristic_direct_candidate_ft"] | audit["supp_direct_upgrade_signal"]
audit["heuristic_strict_sxb_candidate"] = audit["heuristic_strict_sxb_candidate_ft"] | audit["supp_strict_sxb_upgrade_signal"]
audit["heuristic_strict_sxb_possible"] = audit["heuristic_strict_sxb_possible_ft"] | audit["supp_strict_sxb_upgrade_signal"]
audit["stats_variance_signal"] = audit["stats_variance_signal_ft"] | audit["supp_variance_signal"]
audit["stats_n_signal"] = audit["stats_n_signal_ft"] | audit["supp_n_signal"]
audit["stats_ready_proxy"] = audit["stats_variance_signal"] & audit["stats_n_signal"]

audit["heuristic_stress_baseline_candidate"] = audit["heuristic_stress_baseline_candidate_ft"]
audit.loc[(audit["class_final"]=="B") & audit["supp_factorial_grid_signal"],"heuristic_stress_baseline_candidate"] = True
audit["heuristic_biostim_baseline_candidate"] = audit["heuristic_biostim_baseline_candidate_ft"]
audit.loc[(audit["class_final"]=="C") & audit["supp_factorial_grid_signal"],"heuristic_biostim_baseline_candidate"] = True
audit["heuristic_transfer_direct_candidate"] = audit["class_final"].eq("D") & audit["heuristic_direct_candidate"]
audit["heuristic_axb_candidate"] = audit["heuristic_axb_candidate_ft"]
audit["heuristic_axb_strict_possible"] = audit["heuristic_axb_strict_possible_ft"]

for k in COMPILED_OUTCOME:
    ft_col=f"outcome_{k}_ft"
    if ft_col not in audit.columns: audit[ft_col]=False
    audit[f"outcome_{k}"] = audit[ft_col].fillna(False).astype(bool) | audit.get("supp_outcome_labels",pd.Series("",index=audit.index)).fillna("").map(lambda x:k in split_labels(x))

audit["supplement_added_direct"] = (~audit["heuristic_direct_candidate_ft"]) & audit["heuristic_direct_candidate"]
audit["supplement_added_strict_sxb"] = (~audit["heuristic_strict_sxb_candidate_ft"]) & audit["heuristic_strict_sxb_candidate"]
audit["supplement_added_variance"] = (~audit["stats_variance_signal_ft"]) & audit["stats_variance_signal"]
audit["supplement_added_n"] = (~audit["stats_n_signal_ft"]) & audit["stats_n_signal"]
audit["supplement_added_stats_ready"] = (~audit["stats_ready_proxy_ft"]) & audit["stats_ready_proxy"]
audit["machine_supp_registry_mismatch"] = audit["has_machine_supp_bool"] != audit["has_registered_tabular_supp"]

audit.to_csv(EXPORT_DIR / "feasibility_study_level_all_classes.csv", index=False)
display(audit.head(3))

uplift_rows=[]
for cl in ["A","B","C","D","E","X"]:
    sub=audit[audit["class_final"].eq(cl)]
    uplift_rows.append({
        "class":cl,"n_studies":sub["study_id"].nunique(),
        "studies_with_registered_tabular_supp":int(sub["has_registered_tabular_supp"].sum()),
        "direct_ft_only":int(sub["heuristic_direct_candidate_ft"].sum()),
        "direct_after_supp":int(sub["heuristic_direct_candidate"].sum()),
        "direct_uplift_studies":int(sub["supplement_added_direct"].sum()),
        "strict_sxb_ft_only":int(sub["heuristic_strict_sxb_candidate_ft"].sum()),
        "strict_sxb_after_supp":int(sub["heuristic_strict_sxb_candidate"].sum()),
        "strict_sxb_uplift_studies":int(sub["supplement_added_strict_sxb"].sum()),
        "stats_ready_ft_only":int(sub["stats_ready_proxy_ft"].sum()),
        "stats_ready_after_supp":int(sub["stats_ready_proxy"].sum()),
        "stats_ready_uplift_studies":int(sub["supplement_added_stats_ready"].sum()),
        "raw_replicate_supp_signal":int(sub["supp_raw_replicates_possible"].sum()),
    })
supplement_uplift_summary=pd.DataFrame(uplift_rows)
display(supplement_uplift_summary)


## 7. Study-level feasibility table

Ключевые новые поля v4: `stress_domains`, `stress_domain_group`, `disease_labels`, `is_abiotic`, `is_biotic`, `is_fusarium`.


## 8. Роль каждого класса в существующем evidence base

Здесь видно, что B/C/D/E **могут быть полезны**, но они поддерживают другие части проекта и не подменяют A.


In [ ]:
def boolsum(df,col):
    return int(df.get(col,pd.Series(False,index=df.index)).fillna(False).sum())

class_rows=[]
for cl in ["A","B","C","D","E","X"]:
    sub=audit[audit["class_final"].eq(cl)].copy()
    scan_required = cl in FULLTEXT_CLASSES
    parse_ok=sub.get("parse_status",pd.Series(index=sub.index,dtype=str)).isin(["PARSED","CACHE_V5","CACHE_V4_REUSED","CACHE_V3_REUSED","CACHE_V2_REUSED"])
    class_rows.append({
        "class":cl,
        "n_studies":sub["study_id"].nunique(),
        "fulltext_scan_required":scan_required,
        "parsable_fulltexts":int(parse_ok.sum()) if scan_required else np.nan,
        "parse_coverage":float(parse_ok.mean()) if (scan_required and len(sub)) else np.nan,
        "with_supplement":int(sub["has_supplement_bool"].sum()),
        "with_machine_supp_screening":int(sub["has_machine_supp_bool"].sum()),
        "with_registered_tabular_supp":int(sub["has_registered_tabular_supp"].sum()),
        "supp_direct_uplift":int(sub["supplement_added_direct"].sum()),
        "supp_strict_uplift":int(sub["supplement_added_strict_sxb"].sum()),
        "supp_stats_ready_uplift":int(sub["supplement_added_stats_ready"].sum()),
        "direct_candidates":boolsum(sub,"heuristic_direct_candidate"),
        "strict_sxb_candidates":boolsum(sub,"heuristic_strict_sxb_candidate"),
        "stress_baseline_candidates":boolsum(sub,"heuristic_stress_baseline_candidate"),
        "biostim_baseline_candidates":boolsum(sub,"heuristic_biostim_baseline_candidate"),
        "transfer_direct_candidates":boolsum(sub,"heuristic_transfer_direct_candidate"),
        "axb_candidates":boolsum(sub,"heuristic_axb_candidate"),
        "stats_ready_proxy":boolsum(sub,"stats_ready_proxy"),
        "review_secondary_sources":boolsum(sub,"x_review_secondary_source"),
        "meta_analysis_secondary_sources":boolsum(sub,"x_meta_analysis_secondary_source"),
    })
class_summary=pd.DataFrame(class_rows)
display(class_summary)


In [ ]:
A = audit[audit["class_final"].eq("A")].copy()
B = audit[audit["class_final"].eq("B")].copy()
C = audit[audit["class_final"].eq("C")].copy()
D = audit[audit["class_final"].eq("D")].copy()
E = audit[audit["class_final"].eq("E")].copy()
X = audit[audit["class_final"].eq("X")].copy()

stress_labels=list(COMPILED_STRESS)
disease_labels=list(COMPILED_DISEASE)+["other_biotic_disease"]
bio_labels=list(COMPILED_BIO)

def label_summary(df, label_col, labels, prefix):
    rows=[]
    for label in labels:
        mask=df[label_col].map(lambda xs: label in xs if isinstance(xs,list) else False)
        sub=df[mask]
        rows.append({
            prefix:label,
            "n_studies":sub["study_id"].nunique(),
            "direct_candidate":boolsum(sub,"heuristic_direct_candidate"),
            "strict_sxb_candidate":boolsum(sub,"heuristic_strict_sxb_candidate"),
            "stress_baseline_candidate":boolsum(sub,"heuristic_stress_baseline_candidate"),
            "biostim_baseline_candidate":boolsum(sub,"heuristic_biostim_baseline_candidate"),
            "transfer_direct_candidate":boolsum(sub,"heuristic_transfer_direct_candidate"),
            "stats_ready_proxy":boolsum(sub,"stats_ready_proxy"),
            "with_supplement":int(sub["has_supplement_bool"].sum()),
        })
    return pd.DataFrame(rows).sort_values("n_studies",ascending=False)

def domain_summary(df):
    rows=[]
    for name,mask in [
        ("ABIOTIC",df["is_abiotic"]),
        ("BIOTIC_ALL",df["is_biotic"]),
        ("FUSARIUM_SUBGROUP",df["is_fusarium"]),
        ("ABIOTIC+BIOTIC",df["stress_domain_group"].eq("ABIOTIC+BIOTIC")),
    ]:
        sub=df[mask.fillna(False)]
        rows.append({
            "domain":name,"n_studies":sub["study_id"].nunique(),
            "direct_candidate":boolsum(sub,"heuristic_direct_candidate"),
            "strict_sxb_candidate":boolsum(sub,"heuristic_strict_sxb_candidate"),
            "stats_ready_proxy":boolsum(sub,"stats_ready_proxy"),
            "with_supplement":int(sub["has_supplement_bool"].sum()),
        })
    return pd.DataFrame(rows)

stress_summary_A=label_summary(A,"stress_labels",stress_labels,"stress")
stress_summary_B=label_summary(B,"stress_labels",stress_labels,"stress")
stress_summary_D=label_summary(D,"stress_labels",stress_labels,"stress")
disease_summary_A=label_summary(A,"disease_labels",disease_labels,"disease")
disease_summary_B=label_summary(B,"disease_labels",disease_labels,"disease")
disease_summary_D=label_summary(D,"disease_labels",disease_labels,"disease")
domain_summary_A=domain_summary(A)
domain_summary_B=domain_summary(B)
domain_summary_D=domain_summary(D)
bio_summary_A=label_summary(A,"biostim_labels",bio_labels,"biostimulant")
bio_summary_C=label_summary(C,"biostim_labels",bio_labels,"biostimulant")
bio_summary_D=label_summary(D,"biostim_labels",bio_labels,"biostimulant")

print("A — core tomato: domain support")
display(domain_summary_A)
print("A — abiotic stress subtypes")
display(stress_summary_A)
print("A — biotic disease composition (descriptive; pooled for main model)")
display(disease_summary_A)
print("A — biostimulants")
display(bio_summary_A)
print("B — tomato stress baseline")
display(domain_summary_B); display(stress_summary_B); display(disease_summary_B)
print("C — tomato biostimulant-only baseline")
display(bio_summary_C)
print("D — transfer domain")
display(domain_summary_D); display(stress_summary_D); display(disease_summary_D); display(bio_summary_D)


## 9. Core A evidence network and D transfer network

Main model-scope columns are `drought / salinity / heat / cold / BIOTIC pooled`. Fusarium is reported separately as a subgroup rather than being treated as an additional independent stress category.


In [ ]:
def scope_mask(df, scope):
    if scope=="biotic_all":
        return df["is_biotic"].fillna(False)
    return df["stress_labels"].map(lambda xs: scope in xs if isinstance(xs,list) else False)

def evidence_matrix_scope(df, candidate_col=None, scopes=MODEL_SCOPE_CATEGORIES):
    m=pd.DataFrame(0,index=bio_labels,columns=scopes,dtype=int)
    for b in bio_labels:
        bmask=df["biostim_labels"].map(lambda xs:b in xs if isinstance(xs,list) else False)
        for s in scopes:
            mask=bmask & scope_mask(df,s)
            if candidate_col:
                mask &= df.get(candidate_col,pd.Series(False,index=df.index)).fillna(False)
            m.loc[b,s]=df.loc[mask,"study_id"].nunique()
    return m

def evidence_matrix_fusarium(df,candidate_col=None):
    m=pd.DataFrame(0,index=bio_labels,columns=["fusarium"],dtype=int)
    for b in bio_labels:
        mask=df["biostim_labels"].map(lambda xs:b in xs if isinstance(xs,list) else False) & df["is_fusarium"].fillna(False)
        if candidate_col:
            mask &= df.get(candidate_col,pd.Series(False,index=df.index)).fillna(False)
        m.loc[b,"fusarium"]=df.loc[mask,"study_id"].nunique()
    return m

matrix_A=evidence_matrix_scope(A)
matrix_A_direct=evidence_matrix_scope(A,"heuristic_direct_candidate")
matrix_A_strict=evidence_matrix_scope(A,"heuristic_strict_sxb_candidate")
matrix_A_fusarium=evidence_matrix_fusarium(A)
matrix_A_fusarium_direct=evidence_matrix_fusarium(A,"heuristic_direct_candidate")
matrix_D=evidence_matrix_scope(D)
matrix_D_direct=evidence_matrix_scope(D,"heuristic_transfer_direct_candidate")

print("A: expanded scope — all studies")
display(matrix_A)
print("A: expanded scope — direct candidates")
display(matrix_A_direct)
print("A: expanded scope — strict S×B candidates")
display(matrix_A_strict)
print("A: Fusarium subgroup")
display(matrix_A_fusarium)
print("D: transfer evidence")
display(matrix_D)


## 10. Outcome feasibility и quantitative-statistics readiness

TFM указывает terminal dry biomass как primary endpoint, а остальные outcome families — secondary. Full-text text mining здесь является **lower-bound proxy**: figure-only values могут быть пропущены.


In [ ]:
def outcome_summary_for(df, class_name):
    rows=[]
    for outcome in COMPILED_OUTCOME:
        col=f"outcome_{outcome}"
        if col not in df.columns: continue
        sub=df[df[col].fillna(False)]
        rows.append({
            "class":class_name,
            "outcome_family":outcome,
            "n_studies":sub["study_id"].nunique(),
            "direct_candidate":boolsum(sub,"heuristic_direct_candidate"),
            "strict_sxb_candidate":boolsum(sub,"heuristic_strict_sxb_candidate"),
            "stats_ready_proxy":boolsum(sub,"stats_ready_proxy"),
        })
    return pd.DataFrame(rows).sort_values("n_studies",ascending=False)

outcome_A=outcome_summary_for(A,"A")
outcome_B=outcome_summary_for(B,"B")
outcome_C=outcome_summary_for(C,"C")
outcome_D=outcome_summary_for(D,"D")
display(outcome_A)


## 11. X: обзоры и meta-analysis как secondary-source layer

Они **не входят в training N**. Их функция — reference mining, поиск пропущенных первичных исследований, терминология и сопоставление с опубликованными synthesis.


In [ ]:
x_secondary = X[
    X.get("x_review_secondary_source",pd.Series(False,index=X.index)).fillna(False) |
    X.get("x_meta_analysis_secondary_source",pd.Series(False,index=X.index)).fillna(False)
].copy()

x_secondary["secondary_source_type"] = np.select(
    [
        x_secondary.get("x_meta_analysis_secondary_source",pd.Series(False,index=x_secondary.index)).fillna(False),
        x_secondary.get("x_review_secondary_source",pd.Series(False,index=x_secondary.index)).fillna(False),
    ],
    ["META_ANALYSIS","REVIEW"], default="OTHER_SECONDARY"
)
cols=[c for c in ["study_id","title","secondary_source_type","screening_evidence","notes","parse_status"] if c in x_secondary.columns]
x_review_mining_queue=x_secondary[cols].copy()
print("X reviews/meta-analyses potentially useful for reference mining:",len(x_review_mining_queue))
display(x_review_mining_queue.head(50))


## 12. Feasibility A × B между биостимуляторами

В TFM это **условный exploratory/stretch objective**. Порог: ≥40 строгих factorial designs с сопоставимыми дозами. Regex может только сформировать верхнеуровневую candidate queue; подтверждение требует extraction.


In [ ]:
axb_by_class=[]
for cl,df in [("A",A),("C",C),("D",D),("E",E)]:
    axb_by_class.append({
        "class":cl,
        "n_studies":df["study_id"].nunique(),
        "axb_language_candidates":boolsum(df,"heuristic_axb_candidate"),
        "axb_strict_possible":boolsum(df,"heuristic_axb_strict_possible"),
    })
axb_summary=pd.DataFrame(axb_by_class)
display(axb_summary)

axb_tomato_possible = int(
    pd.concat([A,C],ignore_index=True).drop_duplicates("study_id")
      .get("heuristic_axb_strict_possible",pd.Series(dtype=bool)).fillna(False).sum()
)
print("Tomato A/C strict-AxB-possible heuristic upper bound:", axb_tomato_possible)


## 13. Frozen TFM threshold audit + expanded-scope operational audit

Оригинальные Table 4 thresholds **не переписываются**. Для `cold` и `BIOTIC pooled` те же study-level cutoffs применяются как transparent operational rule, чтобы заранее понять, достаточно ли поддержки для pooled moderator / standalone model. Fusarium показывается отдельно как subgroup.


In [ ]:
# Helper counts in class A
n_A=A["study_id"].nunique()
n_A_direct=boolsum(A,"heuristic_direct_candidate")
n_A_strict=boolsum(A,"heuristic_strict_sxb_candidate")
n_A_strict_possible=boolsum(A,"heuristic_strict_sxb_possible")
n_A_stats=boolsum(A,"stats_ready_proxy")
n_D_direct=boolsum(D,"heuristic_transfer_direct_candidate")
n_B_baseline=boolsum(B,"heuristic_stress_baseline_candidate")
n_C_baseline=boolsum(C,"heuristic_biostim_baseline_candidate")

def count_scope(df, scope, direct=False):
    mask=scope_mask(df,scope)
    if direct:
        mask &= df["heuristic_direct_candidate"].fillna(False)
    return int(df.loc[mask,"study_id"].nunique())

A_all_by_scope={s:count_scope(A,s,False) for s in MODEL_SCOPE_CATEGORIES}
A_direct_by_scope={s:count_scope(A,s,True) for s in MODEL_SCOPE_CATEGORIES}
A_all_by_stress={s:count_scope(A,s,False) for s in MAIN_ABIOTIC_STRESSES}
A_direct_by_stress={s:count_scope(A,s,True) for s in MAIN_ABIOTIC_STRESSES}
A_direct_original={s:A_direct_by_stress.get(s,0) for s in ORIGINAL_TFM_STRESSES}

n_A_biotic=int(A.loc[A["is_biotic"].fillna(False),"study_id"].nunique())
n_A_biotic_direct=int(A.loc[A["is_biotic"].fillna(False) & A["heuristic_direct_candidate"].fillna(False),"study_id"].nunique())
n_A_fusarium=int(A.loc[A["is_fusarium"].fillna(False),"study_id"].nunique())
n_A_fusarium_direct=int(A.loc[A["is_fusarium"].fillna(False) & A["heuristic_direct_candidate"].fillna(False),"study_id"].nunique())

threshold_rows=[]

# Original frozen TFM requirement: drought/salinity/heat.
min_original=min(A_direct_original.values()) if A_direct_original else 0
threshold_rows.append({
    "TFM_requirement":"ORIGINAL TFM multistress ML — study count (drought/salinity/heat)",
    "threshold":f">={TFM_THRESHOLDS['multistress_min_studies']} studies total AND >={TFM_THRESHOLDS['multistress_min_studies_per_main_stress']} per original main stress",
    "observed":f"A={n_A}; direct candidates={n_A_direct}; original direct={A_direct_original}",
    "status":"PROVISIONALLY_MEETS_STUDY_THRESHOLD" if (n_A>=TFM_THRESHOLDS['multistress_min_studies'] and min_original>=TFM_THRESHOLDS['multistress_min_studies_per_main_stress']) else "HEURISTIC_GAP_NEEDS_REVIEW",
    "new_publications_implication":"Frozen TFM audit trail only."
})

# Current expanded operational scope.
min_extended=min(A_direct_by_scope.values()) if A_direct_by_scope else 0
threshold_rows.append({
    "TFM_requirement":"EXPANDED scope — drought/salinity/heat/cold + BIOTIC pooled",
    "threshold":f"Operationally >={TFM_THRESHOLDS['multistress_min_studies_per_main_stress']} direct-candidate studies per modeled scope category; total A >={TFM_THRESHOLDS['multistress_min_studies']}",
    "observed":f"A={n_A}; direct by scope={A_direct_by_scope}",
    "status":"PROVISIONALLY_SUPPORTED__EFFECTS_PENDING" if (n_A>=TFM_THRESHOLDS['multistress_min_studies'] and min_extended>=TFM_THRESHOLDS['multistress_min_studies_per_main_stress']) else "TARGETED_GAP_REVIEW_NEEDED",
    "new_publications_implication":"Use pooled BIOTIC. Do not demand separate thresholds for rare pathogen categories."
})

threshold_rows.append({
    "TFM_requirement":"Multistress ML — primary effects",
    "threshold":f">={TFM_THRESHOLDS['multistress_min_primary_effects']} primary effects",
    "observed":"Unknown before structured extraction",
    "status":"PENDING_EXTRACTION",
    "new_publications_implication":"Do not infer effect count from study count; extraction is required."
})

for s in MODEL_SCOPE_CATEGORIES:
    n=A_direct_by_scope.get(s,0)
    label="BIOTIC pooled" if s=="biotic_all" else s
    threshold_rows.append({
        "TFM_requirement":f"Stress/domain-specific model — {label}",
        "threshold":f">={TFM_THRESHOLDS['stress_specific_min_studies']} studies AND >={TFM_THRESHOLDS['stress_specific_min_effects']} effects",
        "observed":f"direct-candidate studies={n}; effects unknown",
        "status":"STUDY_THRESHOLD_PROVISIONALLY_MET__EFFECTS_PENDING" if n>=TFM_THRESHOLDS['stress_specific_min_studies'] else ("MODERATOR_OR_PARTIAL_POOLING" if n>=TFM_THRESHOLDS['multistress_min_studies_per_main_stress'] else "SPARSE_NEEDS_REVIEW"),
        "new_publications_implication":"Standalone model only if threshold and extracted effect count are met; otherwise keep in pooled model."
    })

threshold_rows.append({
    "TFM_requirement":"Fusarium subgroup (reported separately inside BIOTIC)",
    "threshold":f"Diagnostic use of stress-specific rule: >={TFM_THRESHOLDS['stress_specific_min_studies']} studies for a standalone subgroup model; < threshold remains valid subgroup/meta-regression",
    "observed":f"Fusarium studies={n_A_fusarium}; direct candidates={n_A_fusarium_direct}; effects unknown",
    "status":"POTENTIAL_STANDALONE_SUBGROUP__EFFECTS_PENDING" if n_A_fusarium_direct>=TFM_THRESHOLDS['stress_specific_min_studies'] else ("SUBGROUP_META_REGRESSION_OR_DESCRIPTIVE" if n_A_fusarium_direct>=TFM_THRESHOLDS['interaction_pooling_min_studies'] else "SPARSE_SUBGROUP"),
    "new_publications_implication":"No need to balance every disease subtype. Fusarium is reported because it dominates BIOTIC; other diseases stay pooled."
})

threshold_rows += [
    {
        "TFM_requirement":"Predictive S×B interaction",
        "threshold":f">={TFM_THRESHOLDS['sxb_prediction_min_factorial_studies']} strict factorial studies AND >={TFM_THRESHOLDS['sxb_prediction_min_interaction_effects']} interaction effects",
        "observed":f"strict heuristic={n_A_strict}; strict possible={n_A_strict_possible}; effects unknown",
        "status":"PENDING_EXTRACTION_CONFIRMATION" if n_A_strict_possible>=TFM_THRESHOLDS['sxb_prediction_min_factorial_studies'] else "LIKELY_UNDERPOWERED_FOR_PREDICTIVE_SXB",
        "new_publications_implication":"If below threshold after confirmation, retain S×B synthesis/meta-analysis."
    },
    {
        "TFM_requirement":"A×B synergy prediction",
        "threshold":f">={TFM_THRESHOLDS['axb_prediction_min_strict_designs']} strict paired-dose designs",
        "observed":f"tomato A/C strict-possible heuristic={axb_tomato_possible}",
        "status":"PENDING_EXTRACTION_CONFIRMATION" if axb_tomato_possible>=TFM_THRESHOLDS['axb_prediction_min_strict_designs'] else "STRETCH_OBJECTIVE_LIKELY_INSUFFICIENT",
        "new_publications_implication":"Conditional/stretch only."
    },
    {
        "TFM_requirement":"Interaction pooling",
        "threshold":f">={TFM_THRESHOLDS['interaction_pooling_min_studies']} independent strict factorial studies",
        "observed":f"strict heuristic={n_A_strict}",
        "status":"PROVISIONALLY_FEASIBLE" if n_A_strict>=TFM_THRESHOLDS['interaction_pooling_min_studies'] else "PENDING_MANUAL_CONFIRMATION",
        "new_publications_implication":"Pooling can remain feasible even when predictive interaction modeling is not."
    },
    {
        "TFM_requirement":"Temporal validation",
        "threshold":f">={TFM_THRESHOLDS['temporal_min_studies']} test studies and >={TFM_THRESHOLDS['temporal_min_primary_effects']} effects",
        "observed":f"external_test_registry={external_rows}",
        "status":"PENDING_EXTERNAL_TEST" if external_rows==0 else "TEST_REGISTERED_EFFECTS_PENDING",
        "new_publications_implication":"Future 2024+ external test remains separate from current TRAIN_DEV expansion."
    },
]

tfm_threshold_audit=pd.DataFrame(threshold_rows)
display(tfm_threshold_audit)


## 14. Objective-by-objective scorecard — updated expanded scope

The scorecard now evaluates the current project as **ABIOTIC + pooled BIOTIC**, while keeping the original Table 4 rules visible. Fusarium is a reported subgroup, not a required separate model.


In [ ]:
# Search/screening reproducibility proxies
screening_done = int(scr.get("human_status",pd.Series(index=scr.index,dtype=str)).astype(str).str.upper().eq("DONE").sum())
resolved_identity = int(scr.get("identity_status",pd.Series(index=scr.index,dtype=str)).astype(str).str.upper().eq("RESOLVED").sum())
fulltext_present = int(scr.get("has_fulltext",pd.Series(index=scr.index,dtype=str)).astype(str).str.upper().eq("TRUE").sum())
search_saturation_confirmed = False

required_relational_sheets=["studies","experiments","arms","measurements","contrasts"]
schema_present=all(k in DB for k in required_relational_sheets)
extraction_started=nonempty_rows(DB["measurements"],"measurement_id")>0

# Applicability support across the CURRENT model scope.
combo_support=[]
for b in bio_labels:
    for s in MODEL_SCOPE_CATEGORIES:
        n=int(matrix_A.loc[b,s]) if s in matrix_A.columns else 0
        combo_support.append({"biostimulant":b,"stress_or_domain":s,"n_studies":n,"low_evidence":n<TFM_THRESHOLDS['applicability_low_evidence_below_studies']})
combo_support=pd.DataFrame(combo_support)
n_supported_cells=int((~combo_support["low_evidence"]).sum())
n_total_nonempty_cells=int((combo_support["n_studies"]>0).sum())

objective_rows=[
    {
        "objective_id":"O1",
        "TFM_objective":"Reproducible systematic review of quantitative tomato biostimulant studies across abiotic and biotic stress",
        "primary_classes":"A; X-reviews for reference mining",
        "supporting_classes":"B/C mapped near-misses; D/E separate evidence roles",
        "observed":f"screening DONE={screening_done}/{len(scr)}; fulltext={fulltext_present}/{len(scr)}; A biotic={n_A_biotic}; Fusarium={n_A_fusarium}",
        "status":"PARTIALLY_VERIFIED__SATURATION_LOG_NEEDED" if not search_saturation_confirmed else "FEASIBLE",
        "need_new_publications":"ONLY_IF_TARGETED_GAP_OR_SEARCH_SATURATION_REQUIRES",
        "decision":"Current scope: drought/salinity/heat/cold + all diseases pooled; Fusarium reported separately. Rare disease types are not required to reach standalone thresholds."
    },
    {
        "objective_id":"O2",
        "TFM_objective":"Build harmonized experiment/treatment/environment/genotype/endpoint/statistics database",
        "primary_classes":"A",
        "supporting_classes":"B/C/D/E separate relational roles; X secondary only",
        "observed":f"schema_present={schema_present}; extraction_jobs={nonempty_rows(queue,queue.columns[0] if len(queue.columns) else None)}; measurements_extracted={nonempty_rows(DB['measurements'],'measurement_id')}",
        "status":"READY_FOR_EXTRACTION" if schema_present and not extraction_started else "IN_PROGRESS",
        "need_new_publications":"NO_FOR_SCHEMA",
        "decision":"Add stress_domain and disease/pathogen metadata during extraction; disease-specific outcomes remain separate endpoint families."
    },
    {
        "objective_id":"O3",
        "TFM_objective":"Multilevel meta-analysis/meta-regression with dependence control",
        "primary_classes":"A direct effects",
        "supporting_classes":"B/C separate baseline syntheses; D transfer sensitivity",
        "observed":f"A direct={n_A_direct}; stats-ready={n_A_stats}; BIOTIC direct={n_A_biotic_direct}; Fusarium direct={n_A_fusarium_direct}",
        "status":"PROVISIONALLY_FEASIBLE__EFFECT_COUNT_PENDING" if n_A_direct>=TFM_THRESHOLDS['multistress_min_studies'] else "MANUAL_REVIEW_BEFORE_EXPANSION",
        "need_new_publications":"PENDING_EXTRACTION",
        "decision":"Pool disease studies at BIOTIC domain level for primary feasibility; use Fusarium as subgroup/moderator."
    },
    {
        "objective_id":"O4",
        "TFM_objective":"Train predictive models and compare with simple/meta-regression baselines",
        "primary_classes":"A",
        "supporting_classes":"D transfer augmentation only; B/C separate auxiliary models",
        "observed":f"A={n_A}; A direct={n_A_direct}; direct by current scope={A_direct_by_scope}; primary effects unknown",
        "status":"PROVISIONAL__CHECK_TABLE4_AFTER_EXTRACTION",
        "need_new_publications":"TARGETED_ONLY_IF_EXTENDED_SCOPE_CELL_FAILS",
        "decision":"Primary pooled model may include stress_domain/type. Standalone BIOTIC or Fusarium models require their own study/effect support."
    },
    {
        "objective_id":"O5",
        "TFM_objective":"Grouped, temporal and OOD validation without leakage",
        "primary_classes":"A for grouped CV",
        "supporting_classes":"D transfer/OOD; future external test for temporal confirmation",
        "observed":f"A independent studies={n_A}; ABIOTIC={int(A['is_abiotic'].sum())}; BIOTIC={n_A_biotic}; D={D['study_id'].nunique()}; external test={external_rows}",
        "status":"GROUPED_CV_FEASIBLE__TEMPORAL_PENDING_EXTERNAL_TEST" if n_A>=TFM_THRESHOLDS['multistress_min_studies'] else "GROUPED_CV_DATA_LIMITED",
        "need_new_publications":"YES_LATER_FOR_EXTERNAL_TEST__NOT_FOR_CURRENT_TRAINING",
        "decision":"Grouped-by-study CV remains mandatory. Domain-holdout ABIOTIC↔BIOTIC can be an additional diagnostic, not a replacement for external test."
    },
    {
        "objective_id":"O6",
        "TFM_objective":"Calibrated uncertainty and applicability/evidence domain",
        "primary_classes":"A",
        "supporting_classes":"D domain-shift stress tests",
        "observed":f"nonempty biostimulant×scope cells={n_total_nonempty_cells}; cells with >=5 studies={n_supported_cells}",
        "status":"PROVISIONALLY_FEASIBLE__MODEL_DEPENDENT" if n_supported_cells>0 else "SPARSE",
        "need_new_publications":"TARGET_LOW_SUPPORT_REQUIRED_REGIONS_ONLY",
        "decision":"Rare diseases remain inside BIOTIC pooled and should trigger wider uncertainty/OOD warnings rather than forced per-pathogen models."
    },
    {
        "objective_id":"O7a",
        "TFM_objective":"Estimate stress × biostimulant interaction and predict it when threshold permits",
        "primary_classes":"A strict C/S/B/SB",
        "supporting_classes":"D transfer interaction analysis only",
        "observed":f"strict A={n_A_strict}; strict-possible={n_A_strict_possible}; interaction effects unknown",
        "status":"PREDICTION_PENDING_EXTRACTION" if n_A_strict_possible>=TFM_THRESHOLDS['sxb_prediction_min_factorial_studies'] else "META_ANALYSIS_OR_EVIDENCE_MAP_LIKELY",
        "need_new_publications":"OPTIONAL_TARGETED_FACTORIAL_EXPANSION",
        "decision":"Same factorial logic applies to pathogen stress: C=healthy, S=pathogen, B=biostimulant, SB=pathogen+biostimulant."
    },
    {
        "objective_id":"O7b",
        "TFM_objective":"A × B synergy/additivity/antagonism when design permits",
        "primary_classes":"Tomato A/C strict Control-A-B-A+B candidates",
        "supporting_classes":"D/E external evidence only",
        "observed":f"tomato strict-AxB-possible heuristic={axb_tomato_possible}",
        "status":"CONDITIONAL_STRETCH" if axb_tomato_possible<TFM_THRESHOLDS['axb_prediction_min_strict_designs'] else "PENDING_EXTRACTION_CONFIRMATION",
        "need_new_publications":"NOT_MANDATORY_FOR_TFM_COMPLETION",
        "decision":"Retain evidence-map/meta-analysis fallback when <40 confirmed designs."
    },
    {
        "objective_id":"O8",
        "TFM_objective":"SynerPlant Evidence prototype with prediction, uncertainty, evidence density and analogous studies",
        "primary_classes":"A model + evidence database",
        "supporting_classes":"B/C/D context; X review context only",
        "observed":"Depends on O2-O6",
        "status":"DEPENDENT_ON_UPSTREAM_PIPELINE",
        "need_new_publications":"NO_SEPARATE_EXPANSION_REQUIREMENT",
        "decision":"Prototype should expose ABIOTIC/BIOTIC domain, exact disease/pathogen metadata, evidence density and OOD warning."
    },
]
objective_scorecard=pd.DataFrame(objective_rows)
display(objective_scorecard)


## 15. Decision on additional training literature

Expansion decisions use the **current extended scope** (`drought/salinity/heat/cold/BIOTIC pooled`). Fusarium is evaluated as a subgroup. Individual rare diseases are not targets for balancing.


In [ ]:
# Core thesis completion gate: conservative pre-extraction logic.
if n_A < TFM_THRESHOLDS["multistress_min_studies"]:
    thesis_completion_verdict = "EXPANSION_REQUIRED_FOR_CORE_TFM"
elif n_A_direct < TFM_THRESHOLDS["multistress_min_studies"]:
    thesis_completion_verdict = "MANUAL_FULLTEXT_REVIEW_BEFORE_ANY_EXPANSION"
else:
    thesis_completion_verdict = "EXISTING_CORPUS_PROVISIONALLY_SUFFICIENT_FOR_TFM__EXTRACTION_REQUIRED_TO_CONFIRM"

weak_scope_categories = [s for s,n in A_direct_by_scope.items() if n < TFM_THRESHOLDS["multistress_min_studies_per_main_stress"]]
if weak_scope_categories:
    predictive_scope_verdict = "TARGETED_REVIEW_OR_EXPANSION_NEEDED_FOR_EXTENDED_STRESS_SCOPE"
else:
    predictive_scope_verdict = "EXTENDED_SCOPE_STUDY_THRESHOLDS_PROVISIONALLY_OK__EFFECT_COUNTS_PENDING"

expansion=[]
for s in MODEL_SCOPE_CATEGORIES:
    n=A_direct_by_scope.get(s,0)
    display_name="BIOTIC pooled" if s=="biotic_all" else s
    if n < TFM_THRESHOLDS["multistress_min_studies_per_main_stress"]:
        expansion.append({
            "priority":"HIGH_IF_EXTENDED_SCOPE_IS_MANDATORY",
            "target":"A","dimension":"stress_or_domain","category":display_name,
            "current_direct_candidate_studies":n,
            "reason":f"Below operational multistress minimum {TFM_THRESHOLDS['multistress_min_studies_per_main_stress']}",
            "before_search":"Manual review first; for BIOTIC, pool all diseases rather than balancing each pathogen."
        })
    elif n < TFM_THRESHOLDS["stress_specific_min_studies"]:
        expansion.append({
            "priority":"MEDIUM_ONLY_IF_STANDALONE_MODEL_IS_REQUIRED",
            "target":"A","dimension":"standalone_model","category":display_name,
            "current_direct_candidate_studies":n,
            "reason":f"Below stress/domain-specific study threshold {TFM_THRESHOLDS['stress_specific_min_studies']}",
            "before_search":"Keep in pooled all-stress model unless standalone prediction is essential."
        })

# Fusarium is reported separately, but does not need to be expanded merely to balance BIOTIC.
if n_A_fusarium_direct < TFM_THRESHOLDS["stress_specific_min_studies"]:
    expansion.append({
        "priority":"NO_FORCED_EXPANSION__FUSARIUM_SUBGROUP",
        "target":"A-BIOTIC","dimension":"subgroup","category":"Fusarium",
        "current_direct_candidate_studies":n_A_fusarium_direct,
        "reason":"Fusarium is the dominant disease subgroup but need not become a standalone predictive model.",
        "before_search":"Report as subgroup/meta-regression if below threshold; keep other diseases pooled in BIOTIC."
    })

if n_A_strict_possible < TFM_THRESHOLDS["sxb_prediction_min_factorial_studies"]:
    expansion.append({
        "priority":"OPTIONAL_FOR_PREDICTIVE_SXB","target":"A factorial","dimension":"design","category":"C/S/B/SB",
        "current_direct_candidate_studies":n_A_strict_possible,
        "reason":f"Below predictive S×B threshold {TFM_THRESHOLDS['sxb_prediction_min_factorial_studies']}",
        "before_search":"Confirm strict designs manually; include pathogen factorials as valid S×B designs."
    })

if axb_tomato_possible < TFM_THRESHOLDS["axb_prediction_min_strict_designs"]:
    expansion.append({
        "priority":"NO_EXPANSION_REQUIRED_FOR_CORE__STRETCH_ONLY","target":"A/C factorial","dimension":"A_x_B_synergy","category":"Control-A-B-A+B",
        "current_direct_candidate_studies":axb_tomato_possible,
        "reason":f"Below A×B prediction threshold {TFM_THRESHOLDS['axb_prediction_min_strict_designs']}",
        "before_search":"Use evidence map/meta-analysis fallback."
    })

if D["study_id"].nunique() > 0:
    expansion.append({
        "priority":"OPTIONAL_TRANSFER_ENHANCEMENT","target":"D","dimension":"transfer_learning","category":"other crops stress×biostimulant",
        "current_direct_candidate_studies":n_D_direct,
        "reason":"D supports transfer/OOD but does not increase tomato A N",
        "before_search":"Expand only for an explicit transfer-learning objective."
    })

expansion.append({
    "priority":"MANDATORY_LATER_NOT_NOW","target":"EXTERNAL_TEST","dimension":"temporal_validation","category":"post-lock unseen literature",
    "current_direct_candidate_studies":external_rows,
    "reason":"Temporal/confirmatory evaluation requires unseen studies",
    "before_search":"Use 2024+ literature later, after model/preprocessing lock."
})

expansion_priorities=pd.DataFrame(expansion)
print("TFM COMPLETION VERDICT:",thesis_completion_verdict)
print("EXTENDED PREDICTIVE SCOPE:",predictive_scope_verdict)
print("A direct by scope:",A_direct_by_scope)
print("BIOTIC total/direct:",n_A_biotic,n_A_biotic_direct,"| Fusarium total/direct:",n_A_fusarium,n_A_fusarium_direct)
display(expansion_priorities)


## 16. Что B/C/D/E реально добавляют к TFM

Эта таблица отвечает на вопрос, не «загрязняют» ли вспомогательные классы результат. Они не загрязняют его, если используются как отдельные evidence layers.


In [ ]:
class_contribution = pd.DataFrame([
    ["A","Core tomato mitigation/protection effects across ABIOTIC + BIOTIC; meta-analysis; ML; S×B", "YES", "PRIMARY", "Expand only for explicit low-support current-scope cells after audit/extraction"],
    ["B","Tomato stress/pathogen baseline physiology", "NO", "AUXILIARY", "Do not expand unless a separate baseline analysis becomes important"],
    ["C","Tomato biostimulant effect without stress/pathogen; can contain A×B designs", "NO", "AUXILIARY", "Target only for A×B or standalone baseline objective"],
    ["D","Cross-crop transfer/OOD", "NO", "TRANSFER", "Expand only for explicit transfer-learning objective"],
    ["E","Ontology, feature diversity and evidence mapping", "NO", "AUXILIARY_LOW", "No planned expansion"],
    ["X-review/meta","Reference mining, taxonomy, comparison with prior syntheses", "NO", "SECONDARY_SOURCE", "Use to find missed primary papers; never duplicate effects"],
    ["X-other","Excluded/non-primary evidence", "NO", "EXCLUDE", "No expansion"],
],columns=["class","contribution","counts_in_core_A_sample_size","role","expansion_policy"])
display(class_contribution)


## 17. Manual-review queues before any literature expansion

Новый поиск литературы запускается **только после** проверки тех текущих статей, которые могут изменить verdict.


In [ ]:
manual_rows=[]
for _,r in audit.iterrows():
    cl=str(r.get("class_final","")); reasons=[]
    if cl in FULLTEXT_CLASSES and str(r.get("parse_status","")) not in {"PARSED","CACHE_V5","CACHE_V4_REUSED","CACHE_V3_REUSED","CACHE_V2_REUSED"}:
        reasons.append("FULLTEXT_NOT_PARSED")
    if cl=="A" and not bool(r.get("heuristic_direct_candidate",False)):
        reasons.append("A_DIRECT_DESIGN_UNCLEAR")
    if cl=="A" and bool(r.get("is_biotic",False)) and not bool(r.get("heuristic_direct_candidate",False)):
        reasons.append("BIOTIC_DIRECT_DESIGN_NEEDS_CHECK")
    if cl=="A" and bool(r.get("heuristic_strict_sxb_possible",False)) and not bool(r.get("heuristic_strict_sxb_candidate",False)):
        reasons.append("STRICT_SXB_NEEDS_CHECK")
    if cl in {"A","D"} and not bool(r.get("stats_ready_proxy",False)):
        reasons.append("VARIANCE_OR_N_NOT_TEXT_DETECTED")
    if cl in {"A","C"} and bool(r.get("heuristic_axb_candidate",False)):
        reasons.append("AXB_CANDIDATE_NEEDS_ARM_CONFIRMATION")
    if bool(r.get("supplement_added_direct",False)):
        reasons.append("DIRECT_UPGRADED_BY_SUPPLEMENT__VERIFY_ARMS")
    if bool(r.get("supplement_added_strict_sxb",False)):
        reasons.append("STRICT_SXB_UPGRADED_BY_SUPPLEMENT__VERIFY_4_ARMS")
    if bool(r.get("supp_raw_replicates_possible",False)):
        reasons.append("RAW_REPLICATES_IN_SUPPLEMENT__PRIORITIZE_EXTRACTION")
    if bool(r.get("machine_supp_registry_mismatch",False)):
        reasons.append("MACHINE_SUPP_REGISTRY_MISMATCH")
    if cl=="X" and (bool(r.get("x_review_secondary_source",False)) or bool(r.get("x_meta_analysis_secondary_source",False))):
        reasons.append("SECONDARY_SOURCE_REFERENCE_MINING")
    if bool(r.get("manifest_title_crop_mismatch_flag",False)):
        reasons.append("MANIFEST_TITLE_CROP_IDENTITY_MISMATCH")
    if reasons:
        manual_rows.append({
            "study_id":r.get("study_id",""),"class_final":cl,"title":r.get("title",""),
            "stress_domains":r.get("stress_domains",[]),"stress_labels":r.get("stress_labels",[]),
            "disease_labels":r.get("disease_labels",[]),"is_fusarium":r.get("is_fusarium",False),
            "biostim_labels":r.get("biostim_labels",[]),"parse_status":r.get("parse_status",""),
            "reasons":";".join(reasons)
        })
manual_review_queue=pd.DataFrame(manual_rows)
if not manual_review_queue.empty:
    # Put core-A and biotic design questions first.
    manual_review_queue["priority_rank"]=manual_review_queue["reasons"].map(lambda x:0 if "BIOTIC_DIRECT" in x else (1 if "A_DIRECT" in x else 2))
    manual_review_queue=manual_review_queue.sort_values(["priority_rank","class_final","study_id"]).drop(columns="priority_rank")
display(manual_review_queue.head(60))


## 18. Thesis-ready feasibility figures

Figures show the expanded `ABIOTIC + BIOTIC pooled` scope. Disease subtypes are visualized descriptively; only Fusarium receives a dedicated subgroup summary.


In [ ]:
def savefig_multi(fig, stem):
    for ext in ["png","pdf","svg"]:
        kwargs={"bbox_inches":"tight"}
        if ext=="png": kwargs["dpi"]=600
        fig.savefig(FIG_DIR/f"{stem}.{ext}", **kwargs)

# Figure 1 — corpus classes
fig,ax=plt.subplots(figsize=(7.5,4.8))
cc=class_counts.reindex(["A","B","C","D","E","X"])
ax.bar(cc.index,cc.values)
ax.set_xlabel("Screening class"); ax.set_ylabel("Independent studies")
ax.set_title("Existing SynerPlant evidence corpus")
for i,v in enumerate(cc.values): ax.text(i,v,str(int(v)),ha="center",va="bottom",fontsize=9)
fig.tight_layout(); savefig_multi(fig,"fig_feasibility_v4_01_classes"); plt.show()

# Figure 2 — current A scope: all vs direct candidates; Fusarium shown as subgroup.
fig,ax=plt.subplots(figsize=(9.2,5.2))
labels=MODEL_SCOPE_CATEGORIES+["fusarium_subgroup"]
all_vals=[A_all_by_scope[s] for s in MODEL_SCOPE_CATEGORIES]+[n_A_fusarium]
direct_vals=[A_direct_by_scope[s] for s in MODEL_SCOPE_CATEGORIES]+[n_A_fusarium_direct]
x=np.arange(len(labels)); width=.38
ax.bar(x-width/2,all_vals,width,label="All A studies")
ax.bar(x+width/2,direct_vals,width,label="Direct candidates")
ax.set_xticks(x); ax.set_xticklabels(labels,rotation=25,ha="right")
ax.set_ylabel("Independent studies"); ax.set_title("Class A support across expanded stress scope")
ax.legend()
fig.tight_layout(); savefig_multi(fig,"fig_feasibility_v4_02_A_scope"); plt.show()

# Figure 3 — A scope x biostimulant evidence matrix
fig,ax=plt.subplots(figsize=(9.5,7))
im=ax.imshow(matrix_A.values,aspect="auto")
ax.set_xticks(range(len(matrix_A.columns))); ax.set_xticklabels(matrix_A.columns,rotation=45,ha="right")
ax.set_yticks(range(len(matrix_A.index))); ax.set_yticklabels(matrix_A.index)
for i in range(matrix_A.shape[0]):
    for j in range(matrix_A.shape[1]):
        val=int(matrix_A.iloc[i,j])
        if val>0: ax.text(j,i,str(val),ha="center",va="center",fontsize=8)
ax.set_title("Core tomato evidence: stress/domain × biostimulant (class A)")
fig.colorbar(im,ax=ax,label="Independent studies")
fig.tight_layout(); savefig_multi(fig,"fig_feasibility_v4_03_A_matrix"); plt.show()

# Figure 4 — objective status overview
obj=objective_scorecard.copy(); obj["y"]=np.arange(len(obj))[::-1]
fig,ax=plt.subplots(figsize=(11,6))
ax.scatter([1]*len(obj),obj["y"],s=90)
for _,r in obj.iterrows():
    ax.text(1.03,r["y"],f"{r['objective_id']}  {r['TFM_objective']}\n{r['status']}",va="center",fontsize=8)
ax.set_xlim(.95,2.35); ax.set_ylim(-1,len(obj)); ax.axis("off")
ax.set_title("TFM objective-by-objective feasibility status",loc="left")
fig.tight_layout(); savefig_multi(fig,"fig_feasibility_v4_04_objective_scorecard"); plt.show()

# Figure 5 — evidence layers
fig,ax=plt.subplots(figsize=(9,4.8))
vals=class_summary.set_index("class").reindex(["A","B","C","D","E","X"])["n_studies"]
ax.bar(vals.index,vals.values)
ax.set_ylabel("Independent studies")
ax.set_title("Evidence layers retained separately; A defines the core tomato target")
for i,v in enumerate(vals.values): ax.text(i,v,str(int(v)),ha="center",va="bottom",fontsize=9)
fig.tight_layout(); savefig_multi(fig,"fig_feasibility_v4_05_evidence_layers"); plt.show()

# Figure 6 — biotic composition in A (descriptive only)
plot_disease=disease_summary_A[disease_summary_A["n_studies"]>0].sort_values("n_studies",ascending=False)
if len(plot_disease):
    fig,ax=plt.subplots(figsize=(10,5.5))
    ax.bar(plot_disease["disease"],plot_disease["n_studies"])
    ax.set_ylabel("Independent studies")
    ax.set_title("Biotic disease composition within class A (pooled for primary modeling)")
    ax.tick_params(axis="x",rotation=45)
    fig.tight_layout(); savefig_multi(fig,"fig_feasibility_v4_06_biotic_composition"); plt.show()


## 19. Reproducibility spot-check

Это отдельная **heuristic-validation sample**, не финальный quantitative audit из TFM. Финальный audit после extraction должен использовать формулу TFM:

`n_audit = min[N, max(20, ceil(0.15 × N))]`.


In [ ]:
VALIDATION_N = min(72, max(36, math.ceil(0.10*len(audit))))
parts=[]
for cl,g in audit.groupby("class_final"):
    if g.empty: continue
    n=max(2,round(VALIDATION_N*len(g)/len(audit))); n=min(n,len(g))
    parts.append(g.sample(n=n,random_state=20260919))
validation=pd.concat(parts).drop_duplicates("study_id").head(VALIDATION_N).copy()
validation["manual_class_correct"]=""
validation["manual_domain_correct"]=""
validation["manual_disease_correct"]=""
validation["manual_design_correct"]=""
validation["manual_stats_correct"]=""
validation["manual_notes"]=""
cols=[c for c in [
    "study_id","class_final","title","stress_domains","stress_labels","disease_labels","is_fusarium","biostim_labels",
    "heuristic_direct_candidate","heuristic_strict_sxb_candidate","heuristic_axb_candidate","stats_ready_proxy",
    "manual_class_correct","manual_domain_correct","manual_disease_correct","manual_design_correct","manual_stats_correct","manual_notes"
] if c in validation.columns]
validation=validation[cols]
validation.to_csv(EXPORT_DIR/"feasibility_manual_validation_sample_all_classes.csv",index=False)
display(validation.head(30))


## 20. Exports and final decision report v5

New outputs include manifest compatibility/governance, domain support, disease composition, Fusarium subgroup, extended-scope matrices, and supplement uplift.


In [ ]:
# Core exports (v5.1 lightweight-manifest-aligned ABIOTIC + BIOTIC + supplement-aware)
year_governance.to_csv(EXPORT_DIR/"feasibility_year_governance.csv",index=False)
manifest_identity_review.to_csv(EXPORT_DIR/"feasibility_manifest_identity_review.csv",index=False)
known_post_cutoff.to_csv(EXPORT_DIR/"feasibility_known_post2023_seen_records.csv",index=False)
supp_file_audit.to_csv(EXPORT_DIR/"feasibility_supplement_file_audit.csv",index=False)
supp_study.to_csv(EXPORT_DIR/"feasibility_supplement_study_audit.csv",index=False)
supplement_uplift_summary.to_csv(EXPORT_DIR/"feasibility_supplement_uplift_summary.csv",index=False)
class_summary.to_csv(EXPORT_DIR/"feasibility_class_summary.csv",index=False)
CLASS_ROLE.to_csv(EXPORT_DIR/"feasibility_class_role_map.csv",index=False)
class_contribution.to_csv(EXPORT_DIR/"feasibility_class_contribution_to_TFM.csv",index=False)
protocol_alignment.to_csv(EXPORT_DIR/"feasibility_protocol_alignment.csv",index=False)
tfm_threshold_audit.to_csv(EXPORT_DIR/"feasibility_TFM_table4_and_extended_scope_audit.csv",index=False)
objective_scorecard.to_csv(EXPORT_DIR/"feasibility_TFM_objective_scorecard.csv",index=False)
expansion_priorities.to_csv(EXPORT_DIR/"feasibility_expansion_priorities.csv",index=False)
manual_review_queue.to_csv(EXPORT_DIR/"feasibility_manual_review_queue.csv",index=False)
x_review_mining_queue.to_csv(EXPORT_DIR/"feasibility_X_review_reference_mining_queue.csv",index=False)
axb_summary.to_csv(EXPORT_DIR/"feasibility_AxB_by_class.csv",index=False)
combo_support.to_csv(EXPORT_DIR/"feasibility_applicability_support_cells.csv",index=False)

stress_summary_A.to_csv(EXPORT_DIR/"feasibility_abiotic_stress_A.csv",index=False)
stress_summary_B.to_csv(EXPORT_DIR/"feasibility_abiotic_stress_B.csv",index=False)
stress_summary_D.to_csv(EXPORT_DIR/"feasibility_abiotic_stress_D.csv",index=False)
disease_summary_A.to_csv(EXPORT_DIR/"feasibility_biotic_disease_A.csv",index=False)
disease_summary_B.to_csv(EXPORT_DIR/"feasibility_biotic_disease_B.csv",index=False)
disease_summary_D.to_csv(EXPORT_DIR/"feasibility_biotic_disease_D.csv",index=False)
domain_summary_A.to_csv(EXPORT_DIR/"feasibility_stress_domain_A.csv",index=False)
domain_summary_B.to_csv(EXPORT_DIR/"feasibility_stress_domain_B.csv",index=False)
domain_summary_D.to_csv(EXPORT_DIR/"feasibility_stress_domain_D.csv",index=False)
bio_summary_A.to_csv(EXPORT_DIR/"feasibility_biostimulant_A.csv",index=False)
bio_summary_C.to_csv(EXPORT_DIR/"feasibility_biostimulant_C.csv",index=False)
bio_summary_D.to_csv(EXPORT_DIR/"feasibility_biostimulant_D.csv",index=False)
outcome_A.to_csv(EXPORT_DIR/"feasibility_outcomes_A.csv",index=False)

matrix_A.to_csv(EXPORT_DIR/"feasibility_matrix_A_extended_scope_all.csv")
matrix_A_direct.to_csv(EXPORT_DIR/"feasibility_matrix_A_extended_scope_direct.csv")
matrix_A_strict.to_csv(EXPORT_DIR/"feasibility_matrix_A_extended_scope_strict.csv")
matrix_A_fusarium.to_csv(EXPORT_DIR/"feasibility_matrix_A_fusarium_all.csv")
matrix_A_fusarium_direct.to_csv(EXPORT_DIR/"feasibility_matrix_A_fusarium_direct.csv")
matrix_D.to_csv(EXPORT_DIR/"feasibility_matrix_D_extended_scope.csv")
matrix_D_direct.to_csv(EXPORT_DIR/"feasibility_matrix_D_extended_scope_direct.csv")

summary={
    "thesis_completion_verdict":thesis_completion_verdict,
    "predictive_scope_verdict":predictive_scope_verdict,
    "class_counts":{k:int(v) for k,v in class_counts.items()},
    "supplement_uplift":supplement_uplift_summary.to_dict("records"),
    "manifest":{"database_version":DB_LAYER_VERSION,"corpus_version":CORPUS_VERSION,"train_dev_max_year":TRAIN_DEV_MAX_YEAR,"known_post2023_excluded":int(scr_all["known_post_2023"].sum()),"unknown_year_retained":int(scr_all["publication_year_unknown"].sum())},
    "scope_definition":{
        "ABIOTIC":MAIN_ABIOTIC_STRESSES,
        "BIOTIC":"all diseases pooled",
        "reported_subgroup":"Fusarium",
        "rare_diseases":"retained as metadata; no standalone threshold required",
    },
    "A_core":{
        "n_studies":int(n_A),"direct_candidates":int(n_A_direct),
        "strict_sxb_candidates":int(n_A_strict),"strict_sxb_possible":int(n_A_strict_possible),
        "stats_ready_proxy":int(n_A_stats),
        "direct_by_scope":{k:int(v) for k,v in A_direct_by_scope.items()},
        "biotic_studies":int(n_A_biotic),"biotic_direct":int(n_A_biotic_direct),
        "fusarium_studies":int(n_A_fusarium),"fusarium_direct":int(n_A_fusarium_direct),
    },
    "auxiliary":{
        "B_stress_baseline_candidates":int(n_B_baseline),"C_biostim_baseline_candidates":int(n_C_baseline),
        "D_transfer_direct_candidates":int(n_D_direct),"X_review_meta_sources":int(len(x_review_mining_queue)),
        "AxB_tomato_strict_possible":int(axb_tomato_possible),
    },
    "external_test_rows":int(external_rows),"TFM_thresholds":TFM_THRESHOLDS,
}
with open(EXPORT_DIR/"feasibility_summary_v5.json","w",encoding="utf-8") as f:
    json.dump(summary,f,ensure_ascii=False,indent=2)

lines=[]
lines.append("SYNERPLANT TFM — MANIFEST-ALIGNED ABIOTIC + BIOTIC SUPPLEMENT-AWARE FEASIBILITY AUDIT v5")
lines.append("="*72)
lines.append(f"TFM COMPLETION: {thesis_completion_verdict}")
lines.append(f"EXTENDED PREDICTIVE SCOPE: {predictive_scope_verdict}")
lines.append("")
lines.append("CLASS COUNTS: " + ", ".join(f"{k}={int(v)}" for k,v in class_counts.items()))
lines.append(f"MANIFEST: database={DB_LAYER_VERSION}; corpus={CORPUS_VERSION}; <= {TRAIN_DEV_MAX_YEAR} development guard active")
lines.append(f"Known post-2023 already-seen records excluded from feasibility={int(scr_all['known_post_2023'].sum())}; unknown publication year retained={int(scr_all['publication_year_unknown'].sum())}")
lines.append(f"A direct candidates={n_A_direct}; strict SxB={n_A_strict}; strict possible={n_A_strict_possible}; stats-ready={n_A_stats}")
lines.append(f"A direct by current scope: {A_direct_by_scope}")
lines.append(f"BIOTIC pooled: total={n_A_biotic}; direct={n_A_biotic_direct}")
lines.append(f"Fusarium subgroup: total={n_A_fusarium}; direct={n_A_fusarium_direct}")
lines.append("")
lines.append("SUPPLEMENT UPLIFT")
for _,r in supplement_uplift_summary.iterrows():
    lines.append(f"- {r['class']}: tabular_supp_studies={r['studies_with_registered_tabular_supp']}; direct +{r['direct_uplift_studies']}; strict SxB +{r['strict_sxb_uplift_studies']}; stats-ready +{r['stats_ready_uplift_studies']}; raw-replicate signal={r['raw_replicate_supp_signal']}")
lines.append("")
lines.append("OBJECTIVE SCORECARD")
for _,r in objective_scorecard.iterrows():
    lines.append(f"- {r['objective_id']}: {r['status']} | new publications: {r['need_new_publications']}")
    lines.append(f"  {r['decision']}")
lines.append("")
lines.append("EXPANSION PRIORITIES")
for _,r in expansion_priorities.iterrows():
    lines.append(f"- {r['priority']} | {r['target']} | {r['dimension']} | {r['category']} | current={r['current_direct_candidate_studies']}")
    lines.append(f"  Before search: {r['before_search']}")
lines.append("")
lines.append("RULE: BIOTIC diseases are pooled for the main feasibility/model-scope count; Fusarium is reported separately as the dominant subgroup.")
lines.append("RULE: Rare diseases remain exact metadata/moderators; they are not forced into separate predictive models.")
lines.append("RULE: B/C/D/E never increase the core tomato-A sample size. X reviews never create effect-size rows.")
lines.append("RULE: Supplementary can upgrade extractability but never independent-study N.")
lines.append("RULE: A true external temporal test is collected separately after model/preprocessing lock (2024+ later).")
report="\n".join(lines)
print(report)
with open(EXPORT_DIR/"FEASIBILITY_TFM_DECISION_v5.txt","w",encoding="utf-8") as f:
    f.write(report)

print("\nExports:")
for p in sorted(EXPORT_DIR.glob("*")):
    print(" -",p.name)


In [ ]:
# === DOWNLOAD ALL FEASIBILITY TABLES + FIGURES ===

from pathlib import Path
import shutil
from google.colab import files

# Папка результатов именно для v5.1
AUDIT_DIR = Path("/content/drive/MyDrive/TFM_CORPUS_NORMALIZED/07_feasibility_audit_v5_1_light_manifest_aligned")

EXPORT_DIR = AUDIT_DIR / "exports"
FIG_DIR = AUDIT_DIR / "figures"

# Временная папка для объединения результатов
DOWNLOAD_DIR = Path("/content/SynerPlant_feasibility_v5_1_results")

if DOWNLOAD_DIR.exists():
    shutil.rmtree(DOWNLOAD_DIR)

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Копируем таблицы
if EXPORT_DIR.exists():
    shutil.copytree(EXPORT_DIR, DOWNLOAD_DIR / "tables")

# Копируем рисунки
if FIG_DIR.exists():
    shutil.copytree(FIG_DIR, DOWNLOAD_DIR / "figures")

# Создаём ZIP
zip_base = "/content/SynerPlant_feasibility_v5_1_results"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=DOWNLOAD_DIR)

print("Готово:")
print(zip_path)

# Показываем количество файлов
tables_n = len(list((DOWNLOAD_DIR / "tables").glob("*"))) if (DOWNLOAD_DIR / "tables").exists() else 0
figures_n = len(list((DOWNLOAD_DIR / "figures").glob("*"))) if (DOWNLOAD_DIR / "figures").exists() else 0

print(f"Таблиц/отчётов: {tables_n}")
print(f"Рисунков: {figures_n}")

# Скачать на компьютер
files.download(zip_path)

## 21. Как интерпретировать v5.1

Главный результат нужно читать в четырёх слоях:

1. **Manifest governance** — database/corpus version, schema, one-fulltext-per-study, year cutoff и identity flags;
2. **ABIOTIC** — drought, salinity, heat, cold;
3. **BIOTIC pooled** — все disease studies вместе для оценки достаточности и основной pooled modeling strategy;
4. **Fusarium subgroup** — показывается отдельно, потому что он является важным биотическим поддоменом, но не обязан становиться самостоятельной predictive model.

### Temporal rule
Known `year >= 2024` records уже в текущем manifest **не считаются TRAIN_DEV feasibility**. Они также не должны автоматически становиться confirmatory test, если уже были просмотрены. Чистый external test собирается после freeze. Unknown-year records пока остаются в development audit, но должны быть доуточнены до финального temporal lock.

### `EXISTING_CORPUS_PROVISIONALLY_SUFFICIENT_FOR_TFM__EXTRACTION_REQUIRED_TO_CONFIRM`
Study-level evidence выглядит достаточным. Финальное решение зависит от количества подтверждённых direct effects, variance/n и независимых factorial contrasts после structured extraction.

### `TARGETED_REVIEW_OR_EXPANSION_NEEDED_FOR_EXTENDED_STRESS_SCOPE`
Хотя общий A может быть большим, хотя бы один из `drought/salinity/heat/cold/BIOTIC pooled` имеет <8 direct-candidate studies. Сначала ручная проверка, затем точечный добор именно этой ячейки.

### BIOTIC
Не требуйте по 25 исследований отдельно для Fusarium, Botrytis, Rhizoctonia, Ralstonia и т.д. Основная feasibility считается по **BIOTIC pooled**. Exact disease/pathogen остаётся metadata/moderator. Fusarium получает отдельный subgroup report.

### Disease disambiguation
v5.1 не приравнивает любое `damping-off` к Pythium и не присваивает Ralstonia только по слову `bacterial wilt`, если текст уже называет другой pathogen. Комбинированные `early and late blight` и `bacterial speck and spot` получают multi-label.

### Disease outcomes
`disease_severity`, `disease_incidence`, `pathogen_load`, `lesion_symptoms`, `survival_mortality` сохраняются как отдельные outcome families. Их нельзя автоматически смешивать с biomass/yield в один raw target. Benefit-oriented transformation делается позже на этапе effect construction.

### Supplementary
Смотрите:
- `feasibility_supplement_file_audit.csv`;
- `feasibility_supplement_study_audit.csv`;
- `feasibility_supplement_uplift_summary.csv`.

Supplementary **не увеличивают independent N**.

### Manifest QA
Смотрите:
- `feasibility_year_governance.csv`;
- `feasibility_known_post2023_seen_records.csv`;
- `feasibility_manifest_identity_review.csv`.
